In [1]:
# Bloco 1 — Configuração, funções de normalização e gravação segura da tabela
from pathlib import Path
from datetime import datetime
import re
import shutil
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Raiz do projeto = primeira pasta, a partir da do notebook e a subir, que contém data/processed
BASE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "processed").is_dir()), None)
assert BASE is not None, "Pasta do projeto não encontrada: abrir o notebook dentro da pasta do projeto (a que contém data/processed)"
PATH_TAB = BASE / "data" / "processed" / "tabela_completa.csv"
PATH_PRIO = BASE / "data" / "processed" / "imos_prioritarios.csv"
BACKUPS = BASE / "backups"                      # fora do repositório; criada se não existir
BACKUPS.mkdir(exist_ok=True)
for p in (PATH_TAB, PATH_PRIO):
    assert p.exists(), f"Não encontrado: {p}"

# ---------- Normalização de operadores (igual ao Bloco 13 do 02) ----------
NORM_OPS = {
    "CMA-CGM": "CMA CGM", "CMA CGM (Mercosul Line)": "CMA CGM", "Mercosul": "CMA CGM",
    "Mercosul Line": "CMA CGM", "Hyundai": "HMM", "Hyundai (HMM)": "HMM",
    "COSCO SHIPPING Lines": "COSCO", "NYK Line": "ONE Line", "Aliança (ANRM)": "Maersk",
}
SPLIT_OPS = {"ONE Line/Hapag-Lloyd (serviço RBM)": ["ONE Line", "Hapag-Lloyd"]}
REMOVER_OPS = {"Seaspan Corporation"}
GRUPO_OPS = {"Log-In": "MSC", "CNC Line": "CMA CGM", "Gold Star Line": "ZIM", "OOCL": "COSCO"}
GRUPO_TITULAR = [
    (r"\bnyk\b|mitsui o\.?\s?s\.?\s?k|\bmol\b|\bk[\s-]?line\b|kawasaki kisen|ocean network express|^one\b", "ONE Line"),
    (r"alian[cç]a|hamburg s[uü]d|maersk", "Maersk"),
    (r"log-?in", "MSC"),
    (r"mercosul|^cma[\s-]?cgm", "CMA CGM"),
    (r"^cosco", "COSCO"),
    (r"hapag", "Hapag-Lloyd"),
    (r"^msc\b|mediterranean shipping", "MSC"),
    (r"^hmm\b|hyundai merchant", "HMM"),
    (r"^evergreen", "Evergreen"),
    (r"yang ming", "Yang Ming"),
    (r"^zim\b", "ZIM"),
    (r"^pil\b|pacific international lines", "PIL"),
    (r"^oocl\b|orient overseas", "OOCL"),
]

def ops_lista(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    out = []
    for x in [p.strip() for p in str(s).split(",")]:
        for y in SPLIT_OPS.get(x, [x]):
            y = NORM_OPS.get(y, y)
            if y and y not in REMOVER_OPS and y not in out:
                out.append(y)
    return out

def grupo_titular(t):
    if pd.isna(t):
        return t
    for pat, g in GRUPO_TITULAR:
        if re.search(pat, str(t).strip(), flags=re.I):
            return g
    return str(t).strip()

# ---------- Único ponto de escrita na tabela ----------
def gravar(nova, motivo):
    atual = pd.read_csv(PATH_TAB)
    assert len(nova) >= len(atual), f"A nova versão tem menos navios ({len(nova)} < {len(atual)})"
    assert set(atual["IMO"]) <= set(nova["IMO"]), "Há navios da tabela atual que desapareceram"
    assert nova["IMO"].is_unique, "IMO repetido na nova versão"
    alinhado = (nova["Titularidade_Confirmada"] == True) == (nova["Status_Titularidade"] == "Confirmada")
    assert alinhado.all(), f"Status desalinhado em {int((~alinhado).sum())} navios"
    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)
    print(f"GRAVADO — {motivo}")
    print(f"  backup: {bk.name}")
    print(f"  {len(atual)} → {len(nova)} navios | confirmados: {int(atual['Titularidade_Confirmada'].sum())} → "
          f"{int(nova['Titularidade_Confirmada'].sum())}")

tab = pd.read_csv(PATH_TAB)
print(f"Projeto: {BASE.name}")
print(f"tabela_completa: {len(tab)} navios | {tab['Estrutura_Titularidade'].value_counts().to_dict()} "
      f"| confirmados: {int(tab['Titularidade_Confirmada'].sum())}")
print(f"Status_Titularidade: {tab['Status_Titularidade'].value_counts().to_dict()}")

Projeto: ANTAQ_2025_Projeto2
tabela_completa: 324 navios | {'Propria': 243, 'Afretada': 79, 'Afretada Provisoria': 2} | confirmados: 181
Status_Titularidade: {'Confirmada': 181, 'Não confirmada': 143}


In [2]:
# Bloco 1b — Funções e constantes comuns (não grava nada; correr sempre depois do Bloco 1)
TOTAL_TEU_LC_IMO = 11_085_401.9
CLASSES_OK = {"Feeder / Sub-Panamax", "Panamax", "Neo-Panamax", "Large Post-Panamax", "Ultra Large (ULCS)"}
METODO_OBS = "observado — ficha do navio"

def classe_navio(teu):
    if teu < 3000:   return "Feeder / Sub-Panamax"
    if teu < 5000:   return "Panamax"
    if teu < 10000:  return "Neo-Panamax"
    if teu <= 14000: return "Large Post-Panamax"
    return "Ultra Large (ULCS)"

def reefer_v3(teu):
    # teto por faixa de TEU (v3), independente da etiqueta de classe
    teto = 400 if teu < 3000 else 1200 if teu < 10000 else 1500 if teu < 15000 else 2200
    est = round(teu * 0.15)
    if est > teto:
        return teto, f"estimado (NÃO observado) — 15% capado no teto de classe ({teto} TEU)"
    return est, "estimado (NÃO observado) — 15% da capacidade TEU"

def verificar(df):
    conf = df["Titularidade_Confirmada"] == True
    vazio = lambda s: s.isna() | (s.astype(str).str.strip() == "")
    ops = df["Operadores_Confirmados"].fillna("").str.split(r",\s*")
    estr, tipo = df["Estrutura_Titularidade"], df["Tipo_de_Operacao"]
    arm_fora = pd.Series([a not in o and a != "Múltiplos"
                          for a, o in zip(df["Armador_Operador"], ops)], index=df.index)
    return {
        "exclusivo com Linhas_Partilhadas":    (tipo == "exclusivo") & ~vazio(df["Linhas_Partilhadas"]),
        "partilhado sem Linhas_Partilhadas":   (tipo == "partilhado") & vazio(df["Linhas_Partilhadas"]),
        "IMO duplicado":                       df["IMO"].duplicated(keep=False),
        "Afretada sem titular confirmado":     (estr == "Afretada") & ~conf,
        "Provisoria com titular confirmado":   (estr == "Afretada Provisoria") & conf,
        "Classe_Navio vazia":                  df["Classe_Navio"].isna(),
        "Aliança vazia":                       df["Aliança"].isna(),
        "Status desalinhado da booleana":      conf != (df["Status_Titularidade"] == "Confirmada"),
        "Classe fora das 5 etiquetas":         ~df["Classe_Navio"].isin(CLASSES_OK),
        "Armador_Operador fora dos operadores": arm_fora,
    }

print("Funções comuns carregadas (nada gravado).")

Funções comuns carregadas (nada gravado).


In [3]:
# Bloco 2 — MagicPort leva 1: titularidade de 11 navios e notas em 2 por confirmar — REGISTO HISTÓRICO (25/09/2026)
# Idempotente: se já aplicado, não faz nada
TIT = {  # IMO: (titular verificado, estrutura, fonte)
    9526942: ("Maersk A/S", "Propria", "MagicPort 25/09/2026"),
    9285691: ("MSC (Vancouver Shipping Ltd, sociedade no endereço da MSC Shipmanagement)", "Propria", "MagicPort 25/09/2026"),
    9525388: ("Maersk (Moller Singapore AP Pte Ltd)", "Propria", "MagicPort 25/09/2026"),
    9527037: ("Maersk A/S", "Propria", "MagicPort 25/09/2026"),
    9694581: ("Maersk (Moller Singapore AP Pte Ltd; ISM Aliança Navegação)", "Propria", "MagicPort 25/09/2026"),
    9719874: ("Cape Shipping / Cape Maritime (Grey Cedar Shipping Ltd)", "Afretada", "MagicPort 25/09/2026"),
    9697002: ("Mercosul Line Navegação (grupo CMA CGM)", "Propria", "MagicPort 25/09/2026"),
    9770763: ("MSC (Meridian 7 Ltd, sociedade no endereço da MSC)", "Propria", "MagicPort 25/09/2026 + Scheepvaartwest"),
    9332872: ("Hapag-Lloyd Container Linie", "Propria", "EconDB + site Hapag-Lloyd, sem MagicPort; ex-Zim Hamburg"),
    9247546: ("Mount Street Portfolio Advisers (Mad River Shipping Ltd; ISM V.Ships Greece)", "Afretada", "MagicPort 25/09/2026"),
    9850599: ("Evergreen Marine Asia Pte Ltd", "Propria", "MagicPort 25/09/2026"),
}
NOTAS = {
    9850812: "MagicPort 25/09/2026: owner Berlian Steamship Inc (morada Evergreen), ISM Shoei Kisen KK — titularidade inconclusiva",
    9649835: "Site CMA CGM: operador Mercosul Line, bandeira Brasil — registered owner por confirmar",
}

tab = pd.read_csv(PATH_TAB)
assert (set(TIT) | set(NOTAS)) <= set(tab["IMO"]), "IMO fora da tabela"

conf = tab.loc[tab["IMO"].isin(TIT), "Titularidade_Confirmada"] == True
if conf.all():
    print(f"Bloco 2 já aplicado ({len(tab)} navios; os 11 da leva 1 estão confirmados) — nada feito.")
elif conf.any():
    raise AssertionError(f"Estado inesperado: {int(conf.sum())} dos {len(TIT)} navios já confirmados — rever antes de gravar")
else:
    nova = tab.copy()

    def juntar_obs(m, texto):
        obs = nova.loc[m, "Observacoes_Tecnicas"].fillna("").iloc[0]
        if texto not in obs:
            nova.loc[m, "Observacoes_Tecnicas"] = f"{obs} | {texto}" if obs else texto

    for imo, (titular, estr, fonte) in TIT.items():
        m = nova["IMO"] == imo
        if estr == "Propria":
            grupos = {GRUPO_OPS.get(o, o) for o in ops_lista(nova.loc[m, "Operadores_Confirmados"].iloc[0])}
            assert grupo_titular(titular) in grupos, f"{imo}: titular {grupo_titular(titular)} não opera o navio {grupos}"
        nova.loc[m, ["Armador_Titular_Verificado", "Estrutura_Titularidade", "Status_Titularidade"]] = [titular, estr, "Confirmada"]
        nova.loc[m, "Titularidade_Confirmada"] = True
        juntar_obs(m, f"Titularidade: {fonte}")

    for imo, nota in NOTAS.items():
        juntar_obs(nova["IMO"] == imo, nota)

    print(nova.loc[nova["IMO"].isin(list(TIT) + list(NOTAS)),
                   ["IMO", "Nome_Navio", "Armador_Operador", "Armador_Titular_Verificado",
                    "Estrutura_Titularidade", "Titularidade_Confirmada"]].to_string(index=False))
    print()
    gravar(nova, "MagicPort leva 1 — titularidade de 11 navios")
    tab = pd.read_csv(PATH_TAB)
    print(f"Estrutura: {tab['Estrutura_Titularidade'].value_counts().to_dict()}")

Bloco 2 já aplicado (324 navios; os 11 da leva 1 estão confirmados) — nada feito.


In [4]:
# Bloco 3 — Levas 2-3 (grupo 1): 9 navios novos — REGISTO HISTÓRICO (gravado 25/09/2026)
# Idempotente: se os 9 já estão na tabela, não faz nada. O ramo antigo de correção 10→9 (saída do KURE) foi
# retirado: foi aplicado a 25/09 e hoje apagaria o KURE, reinserido no Bloco 13.
CLASSES = [(3000, "Feeder / Sub-Panamax", 400), (5000, "Panamax", None),
           (10000, "Neo-Panamax", 1200), (14001, "Large Post-Panamax", 1500), (1e9, "Ultra Large (ULCS)", 1500)]

def classe_reefer(teu):
    for lim, nome, teto in CLASSES:
        if teu < lim:
            est = round(teu * 0.15)
            if teto is not None and est > teto:
                return nome, teto, f"estimado (NÃO observado) — 15% capado no teto de classe ({teto} TEU)"
            return nome, est, "estimado (NÃO observado) — 15% da capacidade TEU"

FONTE = "Operadores: Flexport Atlas 25/09/2026"
BR = "escala o Brasil no serviço atual (set/2026)"
FORA = "serviço atual (set/2026) fora do Brasil; só o operador da marca, parceiros de 2026 não usados"
# IMO: (nome, TEU, ano, bandeira, LOA, DWT, GT, operadores, armador_operador, rotas, titular, estrutura, confirmada, obs extra)
NOVOS = {
    1016654: ("MSC LEILA", 11586, 2025, "Libéria", 335, 139647, 106463, "MSC", "MSC", "NWCUSASAWC",
              "MSC (Chalune Oceanway Ltd, sociedade no endereço da MSC)", "Propria", True, f"Titularidade: MagicPort 25/09/2026 | {FORA}"),
    1016666: ("MSC INSA", 11586, 2025, "Libéria", 335, 139641, 106463, "MSC", "MSC", "NWCSAEC1",
              "MSC (Niobe Oceanway Ltd, sociedade no endereço da MSC)", "Propria", True, f"Titularidade: MagicPort 25/09/2026 | {FORA} (Hapag-Lloyd ECX em 2026)"),
    9571296: ("LOG-IN ENDURANCE", 2758, 2011, "Brasil", 212, 41568, 35708, "Maersk", "Maersk", "G1F",
              "Log-In Logística, grupo MSC (provável, pelo nome)", "Afretada Provisoria", False, f"{BR}; navio Log-In em serviço Maersk"),
    9603219: ("PEDRO ALVARES CABRAL", 3868, 2013, "Brasil", 227, 52019, 42564, "Maersk, Hapag-Lloyd, Log-In", "Maersk", "G1D/NCF/SMN",
              None, "Propria", False, f"{BR}; Atlas lista ANRM (Aliança, grupo Maersk) e Maersk no G1D"),
    9406623: ("CMA CGM NERVAL", 6554, 2010, "Malta", 299, 89212, 72884, "CMA CGM", "CMA CGM", "KILIMA",
              None, "Propria", False, FORA),
    9603221: ("FERNAO DE MAGALHAES", 3868, 2013, "Brasil", 227, 52072, 42564, "Maersk, Hapag-Lloyd, Log-In, CMA CGM", "Maersk", "G1D/NCF/SMN/LATAM2",
              None, "Propria", False, f"{BR}; Atlas lista ANRM (Aliança, grupo Maersk) e Maersk no G1D"),
    9146479: ("MSC VILDA X", 9578, 1998, "Libéria", 346, 110387, 92198, "MSC", "MSC", "IAS",
              None, "Propria", False, FORA),
    9398228: ("MSC NOA", 6969, 2009, "Libéria", 304, 103192, 76486, "MSC", "MSC", "NWCIPAK",
              None, "Propria", False, f"{FORA} (WEC Lines em 2026)"),
    9625530: ("HMM PREMIUM", 5023, 2013, "Libéria", 255, 63015, 52467, "HMM", "HMM", "FIL",
              None, "Propria", False, BR),
}

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)
assert set(NOVOS) <= set(prio["imo"]), "Há IMOs fora da lista prioritária"

ja = set(NOVOS) & set(tab["IMO"])
if ja == set(NOVOS):
    print(f"Bloco 3 já aplicado ({len(tab)} navios; os 9 do grupo 1 estão na tabela) — nada feito.")
elif ja:
    raise AssertionError(f"Estado inesperado: {len(ja)} dos {len(NOVOS)} navios já na tabela — rever antes de gravar")
else:
    print(f"Estado encontrado: tabela sem estes navios ({len(tab)} navios)")
    linhas = []
    for imo, (nome, teu, ano, band, loa, dwt, gt, ops, arm, rota, titular, estr, conf, obs) in NOVOS.items():
        lista = ops_lista(ops)
        assert arm in lista, f"{imo}: Armador_Operador {arm} não está nos operadores {lista}"
        if conf and estr == "Propria":
            assert grupo_titular(titular) in {GRUPO_OPS.get(o, o) for o in lista}, f"{imo}: titular fora dos operadores"
        classe, reefer, metodo = classe_reefer(teu)
        outros = [o for o in lista if o != arm]
        linhas.append({
            "IMO": imo, "Nome_Navio": nome, "TEU_Capacidade": teu, "TEU_Reefer_Estimado": reefer,
            "Reefer_Metodo": metodo, "Classe_Navio": classe, "Ano_Construcao": ano,
            "Armador_Operador": arm, "Operadores_Confirmados": ", ".join(lista),
            "Tipo_de_Operacao": "partilhado" if outros else "exclusivo", "Aliança": "Não verificada",
            "Linhas_Partilhadas": ", ".join(outros) if outros else None, "Rota_Servico": rota,
            "Bandeira": band, "LOA": loa, "Beam": None, "GT": gt, "DWT": dwt,
            "Armador_Titular_Verificado": titular, "Titularidade_Confirmada": conf,
            "Status_Titularidade": "Confirmada" if conf else "Não confirmada",
            "Estrutura_Titularidade": estr,
            "Observacoes_Tecnicas": " | ".join(x for x in [FONTE, obs] if x),
        })
    novos = pd.DataFrame(linhas).reindex(columns=tab.columns)
    nova = pd.concat([tab, novos], ignore_index=True)

    assert set(tab["IMO"]) <= set(nova["IMO"]), "Saíram navios da tabela"
    assert len(nova) == len(tab) + len(NOVOS)
    falhas = {k: int(v.sum()) for k, v in verificar(nova).items() if v.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)

    tab = pd.read_csv(PATH_TAB)
    ganho = prio.loc[prio["imo"].isin(NOVOS), "teu_lc"].sum()
    print(novos[["IMO", "Nome_Navio", "Classe_Navio", "Armador_Operador", "Operadores_Confirmados",
                 "Estrutura_Titularidade", "Titularidade_Confirmada"]].to_string(index=False))
    print(f"\nGRAVADO — Levas 2-3, grupo 1 (9 navios) | backup: {bk.name}")
    print(f"  TEU LC destes 9 navios: {ganho:,.0f} (+{ganho / TOTAL_TEU_LC_IMO * 100:.2f} pp)")
    print(f"  {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")

Bloco 3 já aplicado (324 navios; os 9 do grupo 1 estão na tabela) — nada feito.


In [5]:
# Bloco 4 — Levas 2-3 (grupo 2): n.º 37, 38, 40 — REGISTO HISTÓRICO (já gravado; se correr, não reescreve)
FONTE = "Operadores: Flexport Atlas 25/09/2026"
BR = "escala o Brasil no serviço atual (set/2026)"

# IMO: (nome, TEU, ano, bandeira, LOA, DWT, GT, Beam, reefer_obs, operadores, armador_operador, rotas,
#       titular, estrutura, confirmada, obs extra)
NOVOS = {
    9720512: ("MSC MICHELA", 9411, 2016, "Portugal", 299, 131034, 95514, None, None,
              "MSC, Hapag-Lloyd, ZIM", "MSC", "WMSAEC/MSE/SAM",
              "MSC (Ample Pointer Ltd, sociedade no endereço da MSC)", "Propria", True,
              f"Titularidade: MagicPort 25/09/2026 | {BR} (WMSAEC); MSE e SAM de parceiros em 2026 — "
              "3 trades, operadores de 2025 não confirmados"),
    9961960: ("LOG-IN EVOLUTION", 3158, 2024, "Brasil", 199, 41750, 34529, None, None,
              "Hapag-Lloyd", "Hapag-Lloyd", "NCF/IRL",
              "Log-In International GmbH (grupo MSC)", "Afretada", True,
              f"Titularidade: MagicPort 25/09/2026 (commercial e ISM Log-In Logística Intermodal) | {BR} "
              "(próximo porto Salvador; destino AIS Navegantes); operador de 2025 pode ter sido Log-In/MSC"),
    9250983: ("MSC DAMLA", 6402, 2002, "Panamá", 293, 83661, 74990, 40.00, 500,
              "MSC, Maersk", "MSC", "GULF-SAEC1/UHD", None, "Propria", False,
              f"{BR} (próximo porto Santos) | Beam e reefer: ficha técnica 28/01/2025 | Titularidade inconclusiva, "
              "rever no fim: registered owner Compania Naviera Andrine S.A. sem morada c/o MSC (anterior Darcey "
              "Shipping c/o MSC); commercial, ISM e operador MSC"),
}

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)
assert set(NOVOS) <= set(prio["imo"]), "Há IMOs fora da lista prioritária"

ja = set(NOVOS) & set(tab["IMO"])
if ja == set(NOVOS):
    print(f"Já gravados ({len(tab)} navios) — nada feito. "
          "Não reescrever: o Bloco 6 atualizou a titularidade destes navios depois.")
elif ja:
    raise AssertionError(f"Estado inesperado: {len(ja)} dos {len(NOVOS)} navios já na tabela")
else:
    print(f"Estado encontrado: tabela sem estes navios ({len(tab)} navios)")
    linhas = []
    for imo, (nome, teu, ano, band, loa, dwt, gt, beam, r_obs, ops, arm, rota,
              titular, estr, conf, obs) in NOVOS.items():
        lista = ops_lista(ops)
        grupos = {GRUPO_OPS.get(o, o) for o in lista}
        assert arm in lista, f"{imo}: Armador_Operador {arm} não está em {lista}"
        if conf and estr == "Propria":
            assert grupo_titular(titular) in grupos, f"{imo}: Propria confirmada com titular fora dos operadores"
        if conf and estr == "Afretada":
            assert grupo_titular(titular) not in grupos, f"{imo}: Afretada confirmada com titular dentro dos operadores"
        assert not (estr == "Afretada Provisoria" and conf), f"{imo}: Provisoria não pode estar confirmada"
        assert not (estr == "Afretada" and not conf), f"{imo}: Afretada exige titular confirmado"

        reefer, metodo = (r_obs, METODO_OBS) if r_obs is not None else reefer_v3(teu)
        outros = [o for o in lista if o != arm]
        linhas.append({
            "IMO": imo, "Nome_Navio": nome, "TEU_Capacidade": teu, "TEU_Reefer_Estimado": reefer,
            "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(teu), "Ano_Construcao": ano,
            "Armador_Operador": arm, "Operadores_Confirmados": ", ".join(lista),
            "Tipo_de_Operacao": "partilhado" if outros else "exclusivo", "Aliança": "Não verificada",
            "Linhas_Partilhadas": ", ".join(outros) if outros else None, "Rota_Servico": rota,
            "Bandeira": band, "LOA": loa, "Beam": beam, "GT": gt, "DWT": dwt,
            "Armador_Titular_Verificado": titular, "Titularidade_Confirmada": conf,
            "Status_Titularidade": "Confirmada" if conf else "Não confirmada",
            "Estrutura_Titularidade": estr,
            "Observacoes_Tecnicas": " | ".join(x for x in [FONTE, obs] if x),
        })
    novos = pd.DataFrame(linhas).reindex(columns=tab.columns)
    nova = (pd.concat([tab, novos.dropna(axis=1, how="all")], ignore_index=True)
              .reindex(columns=tab.columns))
    assert len(nova) == len(tab) + len(NOVOS)
    falhas = {k: int(m.sum()) for k, m in verificar(nova).items() if m.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)
    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(m.sum()) for k, m in verificar(tab).items() if m.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"
    print(f"GRAVADO — {len(NOVOS)} navios | backup: {bk.name} | {len(tab)} navios")

Já gravados (324 navios) — nada feito. Não reescrever: o Bloco 6 atualizou a titularidade destes navios depois.


In [6]:
# Bloco 5 — Levas 2-3 (grupo 3): n.º 42, 43, 44, 45, 46, 47, 49 — REGISTO HISTÓRICO (gravado 25/09/2026)
# Idempotente: se os 7 já estão na tabela, não faz nada (não reescreve — os Blocos 6 e 16 atualizaram
# MSC SILVIA e EVER FAST depois). Usa classe_navio, reefer_v3, verificar, METODO_OBS e TOTAL_TEU_LC_IMO do Bloco 1b.
FONTE = "Operadores: Flexport Atlas 25/09/2026"
BR = "escala o Brasil no serviço atual (set/2026)"
FORA = "serviço atual (set/2026) fora do Brasil; só o operador da marca, parceiros de 2026 não usados"

# IMO: (nome, TEU, ano, bandeira, LOA, DWT, GT, Beam, reefer_obs, operadores, armador_operador, rotas,
#       titular, estrutura, confirmada, obs extra)
NOVOS = {
    9720457: ("MSC SILVIA", 9411, 2015, "Portugal", 299, 131034, 95514, None, None,
              "MSC", "MSC", None, None, "Propria", False,
              f"{FORA}; Atlas sem código de serviço (só MSC como fonte de ETA) | Titularidade inconclusiva, rever no fim: "
              "registered owner Xinlong International Ship, Hong Kong, sem morada c/o MSC; commercial MSC Genebra, "
              "ISM MSC Shipmanagement (MagicPort 25/09/2026)"),
    9506394: ("LOG-IN DISCOVERY", 2546, 2014, "Brasil", 208, 34022, 26374, None, None,
              "Log-In, MSC", "Log-In", "SSV/VITORIA",
              "Log-In Navegação Ltda (grupo MSC)", "Propria", True,
              f"Titularidade: MagicPort 25/09/2026 | {BR} (próximo porto Rio de Janeiro); Log-In e MSC do mesmo grupo"),
    9943085: ("MAERSK REDWOOD", 3500, 2024, "Singapura", 200, 49974, 42799, None, None,
              "Maersk, Hapag-Lloyd", "Maersk", "U4G/CON",
              "Goodwill Maritime Pte Ltd (Singapura)", "Afretada", True,
              "Titularidade: MagicPort 25/09/2026 (owner e commercial Goodwill Maritime, ISM Synergy Pacific) | "
              "Brasil só pelo destino AIS (Rio Grande, MagicPort); Atlas indica próximo porto San Antonio, Chile"),
    9525314: ("MAERSK CAPE COAST", 4496, 2011, "Singapura", 249, 61614, 50940, 37.40, 150,
              "Maersk", "Maersk", "FW2",
              "Maersk (Moller Singapore AP Pte Ltd; registered owner Maersk Line Fleet Management)", "Propria", True,
              f"Titularidade, Beam e reefer: ficha técnica 27/05/2021 | {FORA}"),
    9850874: ("EVER FAST", 11888, 2021, "Libéria", 333, 130573, 116295, None, None,
              "Evergreen", "Evergreen", "NUE", None, "Propria", False,
              f"{FORA} | Titularidade inconclusiva, rever no fim com Ever Fine: registered owner Manatee Navigation SA "
              "c/o Evergreen Taipé, commercial Evergreen, ISM SMTECH Busan (MagicPort 25/09/2026)"),
    9974503: ("MSC GHANA", 8496, 2025, "Libéria", 260, 101456, 79181, None, None,
              "MSC", "MSC", "ENU",
              "MSC (Hauhet Oceanway Ltd, sociedade no endereço da MSC)", "Propria", True,
              f"Titularidade: MagicPort 25/09/2026 | {FORA}"),
    9595448: ("EVER LADEN", 9466, 2012, "Panamá", 334, 103668, 99995, 45.80, 942,
              "Evergreen", "Evergreen", "CIX2",
              "Evergreen (Yamasa New Pulsar V2 SA c/o Evergreen Taipé; ISM e commercial Evergreen)", "Propria", True,
              f"Titularidade, Beam e reefer: ficha técnica 22/03/2025 | {FORA}"),
}

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)
assert set(NOVOS) <= set(prio["imo"]), "Há IMOs fora da lista prioritária"

ja = set(NOVOS) & set(tab["IMO"])
if ja == set(NOVOS):
    print(f"Bloco 5 já aplicado ({len(tab)} navios; os 7 do grupo 3 estão na tabela) — nada feito. "
          "Não reescrever: os Blocos 6 e 16 atualizaram MSC SILVIA e EVER FAST depois.")
elif ja:
    raise AssertionError(f"Estado inesperado: {len(ja)} dos {len(NOVOS)} navios já na tabela — rever antes de gravar")
else:
    print(f"Estado encontrado: tabela sem estes navios ({len(tab)} navios)")
    linhas = []
    for imo, (nome, teu, ano, band, loa, dwt, gt, beam, r_obs, ops, arm, rota,
              titular, estr, conf, obs) in NOVOS.items():
        lista = ops_lista(ops)
        grupos = {GRUPO_OPS.get(o, o) for o in lista}
        assert arm in lista, f"{imo}: Armador_Operador {arm} não está em {lista}"
        if conf and estr == "Propria":
            assert grupo_titular(titular) in grupos, f"{imo}: Propria confirmada com titular fora dos operadores"
        if conf and estr == "Afretada":
            assert grupo_titular(titular) not in grupos, f"{imo}: Afretada confirmada com titular dentro dos operadores"
        assert not (estr == "Afretada Provisoria" and conf), f"{imo}: Provisoria não pode estar confirmada"
        assert not (estr == "Afretada" and not conf), f"{imo}: Afretada exige titular confirmado"

        reefer, metodo = (r_obs, METODO_OBS) if r_obs is not None else reefer_v3(teu)
        outros = [o for o in lista if o != arm]
        linhas.append({
            "IMO": imo, "Nome_Navio": nome, "TEU_Capacidade": teu, "TEU_Reefer_Estimado": reefer,
            "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(teu), "Ano_Construcao": ano,
            "Armador_Operador": arm, "Operadores_Confirmados": ", ".join(lista),
            "Tipo_de_Operacao": "partilhado" if outros else "exclusivo", "Aliança": "Não verificada",
            "Linhas_Partilhadas": ", ".join(outros) if outros else None, "Rota_Servico": rota,
            "Bandeira": band, "LOA": loa, "Beam": beam, "GT": gt, "DWT": dwt,
            "Armador_Titular_Verificado": titular, "Titularidade_Confirmada": conf,
            "Status_Titularidade": "Confirmada" if conf else "Não confirmada",
            "Estrutura_Titularidade": estr,
            "Observacoes_Tecnicas": " | ".join(x for x in [FONTE, obs] if x),
        })
    novos = pd.DataFrame(linhas).reindex(columns=tab.columns)
    nova = (pd.concat([tab, novos.dropna(axis=1, how="all")], ignore_index=True)
              .reindex(columns=tab.columns))
    assert set(tab["IMO"]) <= set(nova["IMO"]), "Saíram navios da tabela"
    assert len(nova) == len(tab) + len(NOVOS)
    falhas = {k: int(m.sum()) for k, m in verificar(nova).items() if m.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)

    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(m.sum()) for k, m in verificar(tab).items() if m.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"
    ganho = prio.loc[prio["imo"].isin(NOVOS), "teu_lc"].sum()
    print(tab.loc[tab["IMO"].isin(NOVOS), ["IMO", "Nome_Navio", "Classe_Navio", "TEU_Reefer_Estimado",
          "Reefer_Metodo", "Armador_Operador", "Operadores_Confirmados", "Tipo_de_Operacao",
          "Estrutura_Titularidade"]].to_string(index=False))
    print(f"\nGRAVADO — Levas 2-3, grupo 3 ({len(NOVOS)} navios) | backup: {bk.name}")
    print(f"  TEU LC destes navios: {ganho:,.0f} (+{ganho / TOTAL_TEU_LC_IMO * 100:.2f} pp)")
    print(f"  {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")
    print("  10 verificações OK no ficheiro gravado")

Bloco 5 já aplicado (324 navios; os 7 do grupo 3 estão na tabela) — nada feito. Não reescrever: os Blocos 6 e 16 atualizaram MSC SILVIA e EVER FAST depois.


In [7]:
# Bloco 6 — Atualização de titularidade de navios já na tabela (acumulativo; seguro correr de novo)
# Usa verificar, ops_lista, GRUPO_OPS e grupo_titular. Só grava se houver mudanças reais.
# MSC ELLEN X saiu daqui: a titularidade foi decidida no Bloco 16.
ANCH = "ISM terceiro não invalida titular c/o linha (precedente MSC Anchorage)"

# IMO: (titular, estrutura esperada, observação) — confirma titularidade
ATUALIZA = {
    9250983: ("MSC (Andrine SA, sociedade no endereço da MSC)", "Propria",
              "Titularidade confirmada: MagicPort 25/09/2026 — registered owner Andrine SA c/o MSC Genebra; "
              "commercial MSC, ISM MSC Shipmanagement"),
    9850812: ("Evergreen (Berlian Steamship Inc c/o Evergreen Taipé)", "Propria",
              "Titularidade confirmada: MagicPort 25/09/2026 — registered owner c/o Evergreen, commercial Evergreen, "
              f"ISM Shoei Kisen; {ANCH}"),
    9850874: ("Evergreen (Manatee Navigation SA c/o Evergreen Taipé)", "Propria",
              "Titularidade confirmada: MagicPort 25/09/2026 — registered owner c/o Evergreen, commercial Evergreen, "
              f"ISM SMTECH Busan; {ANCH}; Econdb owner Evergreen Marine Corp."),
    9649835: ("CMA CGM (CMA CGM SA The French Line, direto)", "Propria",
              "Titularidade confirmada: MagicPort 26/09/2026 (listado como CAP CAPRICORN) — registered owner e "
              "commercial CMA CGM SA Marselha, ISM Mercosul Line | ex-CAP CAPRICORN, renomeado 07/08/2025; "
              "titular de jan-ago/2025 não verificado"),
}
# IMO: (rota, observação)
ROTA = {
    9720457: ("MALABAR", "Rota: Econdb 25/09/2026 (MALABAR EXPRESS); Atlas sem código de serviço"),
}

def juntar_obs(antiga, nova_obs):
    partes = [] if pd.isna(antiga) else [p for p in str(antiga).split(" | ")
                                         if p.strip() and not p.startswith("Titularidade inconclusiva")]
    for p in nova_obs.split(" | "):
        if p not in partes:
            partes.append(p)
    return " | ".join(partes)

tab = pd.read_csv(PATH_TAB)
alvo = set(ATUALIZA) | set(ROTA)
assert alvo <= set(tab["IMO"]), f"IMOs fora da tabela: {alvo - set(tab['IMO'])}"
COLS = ["IMO", "Nome_Navio", "Armador_Titular_Verificado", "Titularidade_Confirmada",
        "Estrutura_Titularidade", "Rota_Servico"]

nova = tab.copy()
for imo, (titular, esperado, obs) in ATUALIZA.items():
    i = nova.index[nova["IMO"] == imo][0]
    grupos = {GRUPO_OPS.get(o, o) for o in ops_lista(nova.at[i, "Operadores_Confirmados"])}
    estr = "Propria" if grupo_titular(titular) in grupos else "Afretada"
    assert estr == esperado, f"{imo}: regra de pertença dá {estr}, esperado {esperado} — nada foi gravado"
    nova.at[i, "Armador_Titular_Verificado"] = titular
    nova.at[i, "Titularidade_Confirmada"] = True
    nova.at[i, "Status_Titularidade"] = "Confirmada"
    nova.at[i, "Estrutura_Titularidade"] = estr
    nova.at[i, "Observacoes_Tecnicas"] = juntar_obs(nova.at[i, "Observacoes_Tecnicas"], obs)

for imo, (rota, obs) in ROTA.items():
    i = nova.index[nova["IMO"] == imo][0]
    nova.at[i, "Rota_Servico"] = rota
    nova.at[i, "Observacoes_Tecnicas"] = juntar_obs(nova.at[i, "Observacoes_Tecnicas"], obs)

if nova.equals(tab):
    print(f"Bloco 6 já aplicado ({len(tab)} navios) — nada feito.")
else:
    assert len(nova) == len(tab) and set(nova["IMO"]) == set(tab["IMO"]), "Número de navios mudou"
    falhas = {k: int(m.sum()) for k, m in verificar(nova).items() if m.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"
    mudou = nova.ne(tab).any(axis=1) & ~(nova.isna() & tab.isna()).all(axis=1)
    print("Linhas alteradas:\n", nova.loc[mudou, COLS].to_string(index=False))
    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)
    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(m.sum()) for k, m in verificar(tab).items() if m.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"
    print(f"\nGRAVADO — Bloco 6 | backup: {bk.name} | {len(tab)} navios | "
          f"confirmados: {int(tab['Titularidade_Confirmada'].sum())}")

Bloco 6 já aplicado (324 navios) — nada feito.


In [8]:
# Bloco 7 — Levas 2-3 (grupo 4): n.º 52, 53, 54, 55, 56, 57, 58 — REGISTO HISTÓRICO (gravado 26/09/2026,
# backup tabela_completa_backup_20260926_181335.csv). Idempotente: se os 7 já estão na tabela, não faz nada
# (não reescreve — os Blocos 14 e 16 atualizaram LOG-IN RESILIENTE e MSC ELLEN X depois).
# Usa classe_navio, reefer_v3, verificar, METODO_OBS e TOTAL_TEU_LC_IMO do Bloco 1b.
FONTE = "Operadores: Flexport Atlas 25-26/09/2026"
BR = "escala o Brasil no serviço atual (set/2026)"
FORA = "serviço atual (set/2026) fora do Brasil; só o operador da marca, parceiros de 2026 não usados"
MP = "Titularidade: MagicPort 26/09/2026"

# IMO: (nome, TEU, ano, bandeira, LOA, DWT, GT, Beam, reefer_obs, operadores, armador_operador, rotas,
#       titular, estrutura, confirmada, obs extra)
NOVOS = {
    9261889: ("CMA CGM FORT ST LOUIS", 2260, 2003, "Chipre", 197, 30804, 26342, 30.20, 556,
              "CMA CGM", "CMA CGM", "AMERXL",
              "CMA CGM (Talent Fortune International Holding Ltd c/o CMA CGM Marselha)", "Propria", True,
              f"Titularidade, Beam e reefer: ficha técnica 30/10/2022 (ISM CMA Ships) | {FORA} "
              "(próximo porto Callao; COSCO NEWS e ONE FLX no Atlas)"),
    9247766: ("MSC TAURUS VII", 6492, 2003, "Chipre", 299, 81171, 75484, None, None,
              "MSC", "MSC", "IROKO",
              "MSC (Gwenora Oceanway Ltd c/o MSC Shipmanagement Limassol)", "Propria", True,
              f"{MP} (commercial e ISM MSC Shipmanagement) | {FORA} (próximo porto Ningbo)"),
    9166780: ("MSC ELLEN X", 9733, 1999, "Libéria", 346, 119448, 92198, None, None,
              "MSC", "MSC", "DRAGON", None, "Propria", False,
              f"{FORA} (próximo porto Nova Iorque) | Titularidade inconclusiva, rever no fim: registered owner "
              "Soroe Shipping Ltd c/o Maersk A/S Copenhaga, commercial e ISM MSC Shipmanagement (MagicPort "
              "26/09/2026) — provável ex-Maersk classe Sovereign vendido à MSC com morada desatualizada; irmão MSC VILDA X"),
    9327669: ("LOG-IN RESILIENTE", 2711, 2006, "Brasil", 210, 38600, 32106, None, None,
              "CMA CGM, Hapag-Lloyd", "Múltiplos", "PLATAMRC/NCF",
              "Log-In (provável, pelo nome) ou Döhle Schiffahrt (fonte secundária) — por confirmar",
              "Afretada Provisoria", False,
              f"{BR} (próximo porto Santos); Atlas lista CMA-CGM e Mercosul (PLATAMRC) e Hapag-Lloyd (NCF/IRL) | "
              "ex-Arosia (2017), MSC Davos (2013), Vasco da Gama (2006); fonte secundária dá owner Döhle — "
              "falta MagicPort"),
    9471898: ("LOG-IN JATOBA", 2814, 2012, "Brasil", 218, 37968, 28554, None, None,
              "CMA CGM, Hapag-Lloyd, Log-In, MSC", "Log-In", "PLATAMRC/NCF/SAS",
              "Log-In Navegação Ltda (grupo MSC)", "Propria", True,
              f"{MP} (commercial e ISM Log-In Logística Intermodal) | {BR} (próximo porto Navegantes); "
              "Atlas lista CMA-CGM e Mercosul (PLATAMRC), Hapag-Lloyd (NCF/IRL), Log-In e MSC (SAS)"),
    9602875: ("SEBASTIAO CABOTO", 3868, 2013, "Brasil", 228, 52065, 42564, None, None,
              "Maersk, Hapag-Lloyd, Log-In", "Maersk", "G1D/NCF/SMN",
              "Aliança Navegação (grupo Maersk)", "Propria", True,
              f"{MP} (owner, commercial e ISM Aliança) | {BR} (próximo porto Pecém); Atlas lista ANRM e Maersk "
              "no G1D; irmão de Pedro Alvares Cabral e Fernao de Magalhaes"),
    9539494: ("MERCOSUL GUARUJA", 3421, 2013, "Brasil", 228, 45878, 39753, None, None,
              "CMA CGM", "CMA CGM", "BRACOMRC",
              "CMA CGM (CMA CGM SA The French Line, direto)", "Propria", True,
              f"{MP} (commercial e ISM Mercosul Line) | {BR} (próximo porto Rio de Janeiro); "
              "Atlas lista CMA-CGM e Mercosul no mesmo serviço (grupo CMA CGM)"),
}

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)
assert set(NOVOS) <= set(prio["imo"]), "Há IMOs fora da lista prioritária"

ja = set(NOVOS) & set(tab["IMO"])
if ja == set(NOVOS):
    print(f"Bloco 7 já aplicado ({len(tab)} navios; os 7 do grupo 4 estão na tabela) — nada feito. "
          "Não reescrever: os Blocos 14 e 16 atualizaram LOG-IN RESILIENTE e MSC ELLEN X depois.")
elif ja:
    raise AssertionError(f"Estado inesperado: {len(ja)} dos {len(NOVOS)} navios já na tabela — rever antes de gravar")
else:
    print(f"Estado encontrado: tabela sem estes navios ({len(tab)} navios)")
    linhas = []
    for imo, (nome, teu, ano, band, loa, dwt, gt, beam, r_obs, ops, arm, rota,
              titular, estr, conf, obs) in NOVOS.items():
        lista = ops_lista(ops)
        grupos = {GRUPO_OPS.get(o, o) for o in lista}
        assert arm in lista or (arm == "Múltiplos" and len(lista) > 1), \
            f"{imo}: Armador_Operador {arm} inválido para {lista}"
        if conf and estr == "Propria":
            assert grupo_titular(titular) in grupos, f"{imo}: Propria confirmada com titular fora dos operadores"
        if conf and estr == "Afretada":
            assert grupo_titular(titular) not in grupos, f"{imo}: Afretada confirmada com titular dentro dos operadores"
        assert not (estr == "Afretada Provisoria" and conf), f"{imo}: Provisoria não pode estar confirmada"
        assert not (estr == "Afretada" and not conf), f"{imo}: Afretada exige titular confirmado"

        reefer, metodo = (r_obs, METODO_OBS) if r_obs is not None else reefer_v3(teu)
        outros = [o for o in lista if o != arm]
        linhas.append({
            "IMO": imo, "Nome_Navio": nome, "TEU_Capacidade": teu, "TEU_Reefer_Estimado": reefer,
            "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(teu), "Ano_Construcao": ano,
            "Armador_Operador": arm, "Operadores_Confirmados": ", ".join(lista),
            "Tipo_de_Operacao": "partilhado" if len(lista) > 1 else "exclusivo", "Aliança": "Não verificada",
            "Linhas_Partilhadas": ", ".join(outros) if len(lista) > 1 else None, "Rota_Servico": rota,
            "Bandeira": band, "LOA": loa, "Beam": beam, "GT": gt, "DWT": dwt,
            "Armador_Titular_Verificado": titular, "Titularidade_Confirmada": conf,
            "Status_Titularidade": "Confirmada" if conf else "Não confirmada",
            "Estrutura_Titularidade": estr,
            "Observacoes_Tecnicas": " | ".join(x for x in [FONTE, obs] if x),
        })
    novos = pd.DataFrame(linhas).reindex(columns=tab.columns)
    nova = (pd.concat([tab, novos.dropna(axis=1, how="all")], ignore_index=True)
              .reindex(columns=tab.columns))
    assert set(tab["IMO"]) <= set(nova["IMO"]), "Saíram navios da tabela"
    assert len(nova) == len(tab) + len(NOVOS)
    falhas = {k: int(m.sum()) for k, m in verificar(nova).items() if m.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)

    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(m.sum()) for k, m in verificar(tab).items() if m.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"
    ganho = prio.loc[prio["imo"].isin(NOVOS), "teu_lc"].sum()
    print(tab.loc[tab["IMO"].isin(NOVOS), ["IMO", "Nome_Navio", "Classe_Navio", "TEU_Reefer_Estimado",
          "Armador_Operador", "Operadores_Confirmados", "Tipo_de_Operacao",
          "Estrutura_Titularidade", "Titularidade_Confirmada"]].to_string(index=False))
    print(f"\nGRAVADO — Levas 2-3, grupo 4 ({len(NOVOS)} navios) | backup: {bk.name}")
    print(f"  TEU LC destes navios: {ganho:,.0f} (+{ganho / TOTAL_TEU_LC_IMO * 100:.2f} pp)")
    print(f"  {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")
    print("  10 verificações OK no ficheiro gravado")

Bloco 7 já aplicado (324 navios; os 7 do grupo 4 estão na tabela) — nada feito. Não reescrever: os Blocos 14 e 16 atualizaram LOG-IN RESILIENTE e MSC ELLEN X depois.


In [9]:
# Bloco 8 — n.º 51 AS SAMANTA: operador inferido por substituição (ANTAQ 2025) — REGISTO HISTÓRICO (gravado 26/09/2026)
# Idempotente: se o navio já está na tabela, não faz nada. Usa classe_navio, reefer_v3, verificar, METODO_OBS e
# TOTAL_TEU_LC_IMO do Bloco 1b.
FONTE = "Operador INFERIDO POR SUBSTITUIÇÃO no loop (dados ANTAQ 2025), não por card de serviço"

# IMO: (nome, TEU, ano, bandeira, LOA, DWT, GT, Beam, reefer_obs, operadores, armador_operador, rotas,
#       titular, estrutura, confirmada, obs extra)
NOVOS = {
    9410260: ("AS SAMANTA", 1713, 2009, "Libéria", 175, 22314, 18123, None, None,
              "Maersk", "Maersk", None,
              "AS Samanta Schiffahrts (c/o Harper Petersen, Hamburgo)", "Afretada", True,
              "Loop Santos/Itapoá ↔ Rio da Prata (Zárate/La Plata/Rosário, trechos para Suape): 2.º navio rodou "
              "9344643 jan-mar → AS SAMANTA abr-jul → MAERSK VENEZIA jun-dez, com AS SABINE o ano todo "
              "(SABINE, VENEZIA e CAPE SOREL confirmados Maersk exclusivo) | 1 escala isolada em fev (2.139 TEU) "
              "por explicar | Atlas 2026: Maersk AM2 + Hapag-Lloyd GUF (próximo Cartagena) | "
              "Titularidade: MagicPort 26/09/2026 (commercial Harper Petersen, ISM Wilhelmsen Ahrenkiel)"),
}

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)
assert set(NOVOS) <= set(prio["imo"]), "Há IMOs fora da lista prioritária"

if set(NOVOS) <= set(tab["IMO"]):
    print(f"Bloco 8 já aplicado ({len(tab)} navios; AS SAMANTA está na tabela) — nada feito.")
else:
    print(f"Estado encontrado: tabela sem este navio ({len(tab)} navios)")
    linhas = []
    for imo, (nome, teu, ano, band, loa, dwt, gt, beam, r_obs, ops, arm, rota,
              titular, estr, conf, obs) in NOVOS.items():
        lista = ops_lista(ops)
        grupos = {GRUPO_OPS.get(o, o) for o in lista}
        assert arm in lista or (arm == "Múltiplos" and len(lista) > 1), f"{imo}: Armador_Operador {arm} inválido"
        if conf and estr == "Propria":
            assert grupo_titular(titular) in grupos, f"{imo}: Propria confirmada com titular fora dos operadores"
        if conf and estr == "Afretada":
            assert grupo_titular(titular) not in grupos, f"{imo}: Afretada confirmada com titular dentro dos operadores"
        assert not (estr == "Afretada Provisoria" and conf) and not (estr == "Afretada" and not conf)

        reefer, metodo = (r_obs, METODO_OBS) if r_obs is not None else reefer_v3(teu)
        outros = [o for o in lista if o != arm]
        linhas.append({
            "IMO": imo, "Nome_Navio": nome, "TEU_Capacidade": teu, "TEU_Reefer_Estimado": reefer,
            "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(teu), "Ano_Construcao": ano,
            "Armador_Operador": arm, "Operadores_Confirmados": ", ".join(lista),
            "Tipo_de_Operacao": "partilhado" if len(lista) > 1 else "exclusivo", "Aliança": "Não verificada",
            "Linhas_Partilhadas": ", ".join(outros) if len(lista) > 1 else None, "Rota_Servico": rota,
            "Bandeira": band, "LOA": loa, "Beam": beam, "GT": gt, "DWT": dwt,
            "Armador_Titular_Verificado": titular, "Titularidade_Confirmada": conf,
            "Status_Titularidade": "Confirmada" if conf else "Não confirmada",
            "Estrutura_Titularidade": estr,
            "Observacoes_Tecnicas": " | ".join(x for x in [FONTE, obs] if x),
        })
    novos = pd.DataFrame(linhas).reindex(columns=tab.columns)
    nova = (pd.concat([tab, novos.dropna(axis=1, how="all")], ignore_index=True)
              .reindex(columns=tab.columns))
    assert set(tab["IMO"]) <= set(nova["IMO"]) and len(nova) == len(tab) + len(NOVOS)
    falhas = {k: int(m.sum()) for k, m in verificar(nova).items() if m.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)
    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(m.sum()) for k, m in verificar(tab).items() if m.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"
    ganho = prio.loc[prio["imo"].isin(NOVOS), "teu_lc"].sum()
    print(f"GRAVADO — Bloco 8 ({len(NOVOS)} navio) | backup: {bk.name} | TEU LC {ganho:,.0f} "
          f"(+{ganho / TOTAL_TEU_LC_IMO * 100:.2f} pp) | {len(tab)} navios")

Bloco 8 já aplicado (324 navios; AS SAMANTA está na tabela) — nada feito.


In [10]:
# Bloco 9 — SPIRIT OF DUBAI (IMO 9366512): ONE, serviço RPX (Rio Grande–Zárate–La Plata)
IMO_N = 9366512
ARM = "ONE Line"   # rótulo existente na tabela (confirmado pela verificação)

# ---- Dados técnicos: Flexport Atlas, ficha do IMO 9366512 (26/09/2026) ----
LOA, DWT, GT = 148, 12485, 10308
BANDEIRA = "Portugal"
assert None not in (LOA, DWT, GT), "Preencher LOA, DWT e GT a partir do Atlas antes de correr"

TEU, ANO = 1147, 2009
TITULAR = "Temeggong Shipping (c/o Eastaway Ship Management, Singapura)"
FONTE_9 = ("Operador: anúncio do serviço RPX da ONE com este navio (Linerlytica, início 8/mai/2025) "
           "+ coerência ANTAQ 2025 (27 escalas LC mai-dez, 98% Rio Grande↔Zárate/La Plata) | "
           "Dados técnicos: Flexport Atlas 26/09/2026")
OBS_9 = ("Titularidade: registered owner Temeggong Shipping, ISM+commercial Eastaway Ship Management "
         "(Singapura) | Atlas 2026 (GUYANA/NSEC/GSE/IRL) NÃO usado para operador: não reflete 2025 | "
         "AIS do Atlas desatualizado (última ETA jul/2026) | Nome EA BREEZE segundo outra fonte, "
         "não confirmado no Atlas | Continuidade até dez/2025 inferida pela ANTAQ, não por fonte da ONE")

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)
assert IMO_N in set(prio["imo"]), "IMO fora da lista prioritária"

rotulos = set(tab["Armador_Operador"].dropna())
assert ARM in rotulos, (f"Rótulo '{ARM}' não existe na tabela. Candidatos: "
                        f"{sorted(r for r in rotulos if 'ONE' in r.upper() or 'OCEAN' in r.upper())}")

if IMO_N in set(tab["IMO"]):
    print(f"Já gravado ({len(tab)} navios) — nada feito.")
else:
    print(f"Estado encontrado: tabela sem este navio ({len(tab)} navios)")
    lista = ops_lista(ARM)
    grupos = {GRUPO_OPS.get(o, o) for o in lista}
    assert grupo_titular(TITULAR) not in grupos, "Afretada confirmada com titular dentro dos operadores"

    reefer, metodo = reefer_v3(TEU)
    linha = pd.DataFrame([{
        "IMO": IMO_N, "Nome_Navio": "SPIRIT OF DUBAI", "TEU_Capacidade": TEU,
        "TEU_Reefer_Estimado": reefer, "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(TEU),
        "Ano_Construcao": ANO, "Armador_Operador": ARM, "Operadores_Confirmados": ", ".join(lista),
        "Tipo_de_Operacao": "exclusivo", "Aliança": "Não verificada", "Linhas_Partilhadas": None,
        "Rota_Servico": "RPX", "Bandeira": BANDEIRA, "LOA": LOA, "Beam": None, "GT": GT, "DWT": DWT,
        "Armador_Titular_Verificado": TITULAR, "Titularidade_Confirmada": True,
        "Status_Titularidade": "Confirmada", "Estrutura_Titularidade": "Afretada",
        "Observacoes_Tecnicas": f"{FONTE_9} | {OBS_9}",
    }]).reindex(columns=tab.columns)

    nova = (pd.concat([tab, linha.dropna(axis=1, how="all")], ignore_index=True)
              .reindex(columns=tab.columns))
    assert set(tab["IMO"]) <= set(nova["IMO"]) and len(nova) == len(tab) + 1
    falhas = {k: int(m.sum()) for k, m in verificar(nova).items() if m.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)

    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(m.sum()) for k, m in verificar(tab).items() if m.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"

    ganho = prio.loc[prio["imo"] == IMO_N, "teu_lc"].sum()
    print(tab.loc[tab["IMO"] == IMO_N, ["IMO", "Nome_Navio", "Classe_Navio", "TEU_Reefer_Estimado",
          "Armador_Operador", "Tipo_de_Operacao", "Rota_Servico", "Estrutura_Titularidade",
          "Titularidade_Confirmada"]].to_string(index=False))
    print(f"\nGRAVADO — Bloco 9 (1 navio) | backup: {bk.name}")
    print(f"  TEU LC: {ganho:,.0f} (+{ganho / TOTAL_TEU_LC_IMO * 100:.2f} pp)")
    print(f"  {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")
    print("  10 verificações OK no ficheiro gravado")

Já gravado (324 navios) — nada feito.


In [11]:
# Bloco 10 — MIRADOR EXPRESS (IMO 9243174): Hapag-Lloyd, serviço SAT (Paranaguá–Itajaí–Santos–Durban–Luanda)
IMO_N = 9243174
ARM = "Hapag-Lloyd"

# ---- Dados técnicos: Flexport Atlas, ficha do IMO 9243174 (26/09/2026) ----
LOA, DWT, GT = 243, 40478, 40146
BANDEIRA = "Libéria"
TEU, ANO = 3237, 2002
TITULAR = "GG Holdings VII LLC (c/o Conbulk Shipmanagement, Pireu)"
FONTE_10 = ("Operador: Atlas Past Port Calls jul-dez/2025 (só escalas do SAT: Paranaguá, Itajaí, Santos, "
            "Durban, Luanda) + rotação SAT publicada pela Hapag-Lloyd (Santos retirado a partir de nov/2025, "
            "confirmado no histórico) + coerência ANTAQ 2025 (26 escalas LC fev-dez, Luanda→Paranaguá 23%) | "
            "Dados técnicos: Flexport Atlas 26/09/2026")
OBS_10 = ("Titularidade: MagicPort 26/09/2026 — registered owner GG Holdings VII LLC, ISM+commercial Conbulk "
          "Shipmanagement (Pireu); ficha antiga sem data dá owner/manager Hapag-Lloyd USA (ex-YORKTOWN EXPRESS "
          "2021) — venda com data desconhecida: se ocorreu em 2026, em 2025 era Propria | Atlas 2026 (Hapag MIA + "
          "ONE AIM, Médio Oriente-Índia-África) NÃO usado para operador | Horários Maersk jan/abr 2025 = "
          "compra de slots provável, não registada como operador")

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)
assert IMO_N in set(prio["imo"]), "IMO fora da lista prioritária"

rotulos = set(tab["Armador_Operador"].dropna())
assert ARM in rotulos, (f"Rótulo '{ARM}' não existe na tabela. Candidatos: "
                        f"{sorted(r for r in rotulos if 'HAPAG' in r.upper())}")

if IMO_N in set(tab["IMO"]):
    print(f"Já gravado ({len(tab)} navios) — nada feito.")
else:
    print(f"Estado encontrado: tabela sem este navio ({len(tab)} navios)")
    lista = ops_lista(ARM)
    grupos = {GRUPO_OPS.get(o, o) for o in lista}
    assert grupo_titular(TITULAR) not in grupos, "Afretada confirmada com titular dentro dos operadores"

    reefer, metodo = reefer_v3(TEU)
    linha = pd.DataFrame([{
        "IMO": IMO_N, "Nome_Navio": "MIRADOR EXPRESS", "TEU_Capacidade": TEU,
        "TEU_Reefer_Estimado": reefer, "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(TEU),
        "Ano_Construcao": ANO, "Armador_Operador": ARM, "Operadores_Confirmados": ", ".join(lista),
        "Tipo_de_Operacao": "exclusivo", "Aliança": "Não verificada", "Linhas_Partilhadas": None,
        "Rota_Servico": "SAT", "Bandeira": BANDEIRA, "LOA": LOA, "Beam": None, "GT": GT, "DWT": DWT,
        "Armador_Titular_Verificado": TITULAR, "Titularidade_Confirmada": True,
        "Status_Titularidade": "Confirmada", "Estrutura_Titularidade": "Afretada",
        "Observacoes_Tecnicas": f"{FONTE_10} | {OBS_10}",
    }]).reindex(columns=tab.columns)

    nova = (pd.concat([tab, linha.dropna(axis=1, how="all")], ignore_index=True)
              .reindex(columns=tab.columns))
    assert set(tab["IMO"]) <= set(nova["IMO"]) and len(nova) == len(tab) + 1
    falhas = {k: int(m.sum()) for k, m in verificar(nova).items() if m.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)

    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(m.sum()) for k, m in verificar(tab).items() if m.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"

    ganho = prio.loc[prio["imo"] == IMO_N, "teu_lc"].sum()
    print(tab.loc[tab["IMO"] == IMO_N, ["IMO", "Nome_Navio", "Classe_Navio", "TEU_Reefer_Estimado",
          "Armador_Operador", "Tipo_de_Operacao", "Rota_Servico", "Estrutura_Titularidade",
          "Titularidade_Confirmada"]].to_string(index=False))
    print(f"\nGRAVADO — Bloco 10 (1 navio) | backup: {bk.name}")
    print(f"  TEU LC: {ganho:,.0f} (+{ganho / TOTAL_TEU_LC_IMO * 100:.2f} pp)")
    print(f"  {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")
    print("  10 verificações OK no ficheiro gravado")

Já gravado (324 navios) — nada feito.


In [12]:
# Bloco 11 — DUSSELDORF EXPRESS (IMO 9143556): Hapag-Lloyd CON + Maersk U4G (ECSA ↔ WCSA)
IMO_N = 9143556
OPS = "Hapag-Lloyd, Maersk"
ARM = "Hapag-Lloyd"

# ---- Dados técnicos: Flexport Atlas, ficha do IMO 9143556 (26/09/2026); Beam do site Hapag-Lloyd ----
LOA, DWT, GT, BEAM = 293, 66525, 53523, 32
BANDEIRA = "Bermudas"
TEU, ANO = 4612, 1998
TITULAR = "Hapag-Lloyd (Hapag-Lloyd AG, Hamburgo)"
FONTE_11 = ("Operadores: Flexport Atlas 26/09/2026 (Hapag-Lloyd CON + Maersk U4G) — regra 1 validada pelo "
            "histórico de escalas: mesmo loop ECSA↔WCSA (Itapoá, Rio, Santos, Rio Grande / Callao, Guayaquil, "
            "Arica, Antofagasta, San Antonio, San Vicente) de jun/2025 a set/2026; ANTAQ jan-nov/2025 com o mesmo "
            "padrão (Santos→Callao 17%) | Dados técnicos: Flexport Atlas 26/09/2026; Beam: site Hapag-Lloyd")
OBS_11 = ("Titularidade: MagicPort 26/09/2026 — registered owner e commercial Hapag-Lloyd AG, ISM Anglo-Eastern "
          "Germany (ISM terceiro não invalida — precedente MSC Anchorage); ficha antiga concorda | Parceiro antes "
          "de jun/2025 não verificado (histórico Atlas começa em jun/2025) | Reefer: site Hapag indica 0 ligações "
          "(implausível, campo provavelmente vazio) — mantido o estimado v3 | Atlas regista mais 1 escala BR por "
          "viagem do que a ANTAQ (ex.: Rio Grande 4/out/2025 ausente na ANTAQ) — a verificar")

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)
assert IMO_N in set(prio["imo"]), "IMO fora da lista prioritária"

lista = ops_lista(OPS)
universo = set(tab["Armador_Operador"].dropna()) | {
    o for s in tab["Operadores_Confirmados"].dropna() for o in ops_lista(s)}
em_falta = [o for o in lista if o not in universo]
assert not em_falta, (f"Rótulos inexistentes na tabela: {em_falta}. Candidatos: "
                      f"{sorted(r for r in universo if any(k in r.upper() for k in ('HAPAG', 'MAERSK')))}")
assert ARM in lista

if IMO_N in set(tab["IMO"]):
    print(f"Já gravado ({len(tab)} navios) — nada feito.")
else:
    print(f"Estado encontrado: tabela sem este navio ({len(tab)} navios)")
    grupos = {GRUPO_OPS.get(o, o) for o in lista}
    assert grupo_titular(TITULAR) in grupos, "Propria confirmada com titular fora dos operadores"

    reefer, metodo = reefer_v3(TEU)
    outros = [o for o in lista if o != ARM]
    linha = pd.DataFrame([{
        "IMO": IMO_N, "Nome_Navio": "DUSSELDORF EXPRESS", "TEU_Capacidade": TEU,
        "TEU_Reefer_Estimado": reefer, "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(TEU),
        "Ano_Construcao": ANO, "Armador_Operador": ARM, "Operadores_Confirmados": ", ".join(lista),
        "Tipo_de_Operacao": "partilhado", "Aliança": "Não verificada",
        "Linhas_Partilhadas": ", ".join(outros), "Rota_Servico": "CON/U4G",
        "Bandeira": BANDEIRA, "LOA": LOA, "Beam": BEAM, "GT": GT, "DWT": DWT,
        "Armador_Titular_Verificado": TITULAR, "Titularidade_Confirmada": True,
        "Status_Titularidade": "Confirmada", "Estrutura_Titularidade": "Propria",
        "Observacoes_Tecnicas": f"{FONTE_11} | {OBS_11}",
    }]).reindex(columns=tab.columns)

    nova = (pd.concat([tab, linha.dropna(axis=1, how="all")], ignore_index=True)
              .reindex(columns=tab.columns))
    assert set(tab["IMO"]) <= set(nova["IMO"]) and len(nova) == len(tab) + 1
    falhas = {k: int(m.sum()) for k, m in verificar(nova).items() if m.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)

    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(m.sum()) for k, m in verificar(tab).items() if m.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"

    ganho = prio.loc[prio["imo"] == IMO_N, "teu_lc"].sum()
    print(tab.loc[tab["IMO"] == IMO_N, ["IMO", "Nome_Navio", "Classe_Navio", "TEU_Reefer_Estimado",
          "Armador_Operador", "Operadores_Confirmados", "Tipo_de_Operacao", "Rota_Servico",
          "Estrutura_Titularidade", "Titularidade_Confirmada"]].to_string(index=False))
    print(f"\nGRAVADO — Bloco 11 (1 navio) | backup: {bk.name}")
    print(f"  TEU LC: {ganho:,.0f} (+{ganho / TOTAL_TEU_LC_IMO * 100:.2f} pp)")
    print(f"  {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")
    print("  10 verificações OK no ficheiro gravado")

Já gravado (324 navios) — nada feito.


In [13]:
# Bloco 12 — GREENLAND (MSC, EUA↔ECSA) + registo do descarte do GREEN SALVADOR
from pathlib import Path

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)

# ================= Parte A — GREENLAND (IMO 9970002) =================
IMO_N = 9970002
ARM = "MSC"
LOA, DWT, GT = 272, 93747, 72805          # Flexport Atlas 26/09/2026
BANDEIRA, TEU, ANO = "Malta", 8010, 2024
TITULAR = "Danaos (Teushipper No 2 Corp c/o Danaos Shipping, Pireu)"
FONTE_12 = ("Operador: Atlas Past Port Calls jun-jul/2025 (Santos, Buenos Aires, Montevideo, Rio Grande, Paranaguá, "
            "Itajaí, Santos, Rio, Salvador, Colón, Freeport, costa leste EUA = rotação publicada do serviço MSC "
            "EUA↔ECSA) + ANTAQ 2025: 21 escalas LC jan-jul, pares de carga EUA↔ECSA (New York→Santos 8,7%, "
            "Norfolk→Santos, Itajaí/Santos→Cristobal); nenhuma escala após jul nas duas fontes | "
            "Dados técnicos: Flexport Atlas 26/09/2026")
OBS_12 = ("Titularidade: registered owner Teushipper No 2 Corp c/o Danaos (Pireu), ISM+commercial Danaos | "
          "Mesmo serviço MSC do KURE em 2025 | Compra de slots por outras linhas neste serviço não verificada | "
          "Desde ago/2025 fora do Brasil (Médio Oriente/Índia–EUA)")

assert IMO_N in set(prio["imo"]), "GREENLAND fora da lista prioritária"
assert ARM in set(tab["Armador_Operador"].dropna()), f"Rótulo '{ARM}' não existe na tabela"

if IMO_N in set(tab["IMO"]):
    print(f"A) GREENLAND já gravado ({len(tab)} navios) — nada feito.")
else:
    print(f"A) Estado encontrado: tabela sem GREENLAND ({len(tab)} navios)")
    lista = ops_lista(ARM)
    grupos = {GRUPO_OPS.get(o, o) for o in lista}
    assert grupo_titular(TITULAR) not in grupos, "Afretada confirmada com titular dentro dos operadores"

    reefer, metodo = reefer_v3(TEU)
    linha = pd.DataFrame([{
        "IMO": IMO_N, "Nome_Navio": "GREENLAND", "TEU_Capacidade": TEU,
        "TEU_Reefer_Estimado": reefer, "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(TEU),
        "Ano_Construcao": ANO, "Armador_Operador": ARM, "Operadores_Confirmados": ", ".join(lista),
        "Tipo_de_Operacao": "exclusivo", "Aliança": "Não verificada", "Linhas_Partilhadas": None,
        "Rota_Servico": "USA-SAEC", "Bandeira": BANDEIRA, "LOA": LOA, "Beam": None, "GT": GT, "DWT": DWT,
        "Armador_Titular_Verificado": TITULAR, "Titularidade_Confirmada": True,
        "Status_Titularidade": "Confirmada", "Estrutura_Titularidade": "Afretada",
        "Observacoes_Tecnicas": f"{FONTE_12} | {OBS_12}",
    }]).reindex(columns=tab.columns)

    nova = (pd.concat([tab, linha.dropna(axis=1, how="all")], ignore_index=True)
              .reindex(columns=tab.columns))
    assert set(tab["IMO"]) <= set(nova["IMO"]) and len(nova) == len(tab) + 1
    falhas = {k: int(m.sum()) for k, m in verificar(nova).items() if m.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)

    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(m.sum()) for k, m in verificar(tab).items() if m.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"

    ganho = prio.loc[prio["imo"] == IMO_N, "teu_lc"].sum()
    print(tab.loc[tab["IMO"] == IMO_N, ["IMO", "Nome_Navio", "Classe_Navio", "TEU_Reefer_Estimado",
          "Armador_Operador", "Tipo_de_Operacao", "Rota_Servico", "Estrutura_Titularidade",
          "Titularidade_Confirmada"]].to_string(index=False))
    print(f"   GRAVADO | backup: {bk.name} | TEU LC {ganho:,.0f} (+{ganho / TOTAL_TEU_LC_IMO * 100:.2f} pp)")
    print(f"   {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()} | 10 verificações OK")

# ================= Parte B — descarte do GREEN SALVADOR (IMO 9976460) =================
PATH_DESC = Path(PATH_TAB).parent / "navios_descartados.csv"
IMO_D = 9976460
desc = pd.read_csv(PATH_DESC)
assert list(desc.columns) == ["IMO", "Nome_Navio", "Tipo_Real", "Motivo", "Data"], f"Formato inesperado: {list(desc.columns)}"
assert IMO_D in set(prio["imo"]), "GREEN SALVADOR fora da lista prioritária"
assert IMO_D not in set(tab["IMO"]), "GREEN SALVADOR está na tabela — não pode ser descartado sem o retirar primeiro"

if IMO_D in set(desc["IMO"]):
    print(f"\nB) GREEN SALVADOR já registado em {PATH_DESC.name} — nada feito.")
else:
    teu_d = prio.loc[prio["imo"] == IMO_D, "teu_lc"].sum()
    reg = pd.DataFrame([{
        "IMO": IMO_D, "Nome_Navio": "GREEN SALVADOR",
        "Tipo_Real": "General Cargo Ship (MagicPort)",
        "Motivo": (f"Não é porta-contentores (tipo prevalece sobre TEU); TEU LC 2025 = {teu_d:,.0f} "
                   f"({teu_d / TOTAL_TEU_LC_IMO * 100:.2f} pp), mantido no denominador; owner Sea 263 Leasing "
                   "c/o COSCO Shipping Specialized Carriers; decisão de Hugo 26/09/2026"),
        "Data": f"{datetime.now():%Y-%m-%d}",
    }])
    bk_d = BACKUPS / f"navios_descartados_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_DESC, bk_d)
    pd.concat([desc, reg], ignore_index=True).to_csv(PATH_DESC, index=False)
    chk = pd.read_csv(PATH_DESC)
    assert len(chk) == len(desc) + 1 and IMO_D in set(chk["IMO"])
    print(f"\nB) GREEN SALVADOR registado em {PATH_DESC.name} ({len(chk)} linha(s)) | backup: {bk_d.name}")
    print(f"   TEU LC {teu_d:,.0f} ({teu_d / TOTAL_TEU_LC_IMO * 100:.2f} pp) — fica no denominador, fora da cobertura")

A) GREENLAND já gravado (324 navios) — nada feito.

B) GREEN SALVADOR já registado em navios_descartados.csv — nada feito.


In [14]:
# Bloco 13 — RDO ENDEAVOUR, SEASPAN EMPIRE (consórcio Golfo↔ECSA, "Múltiplos") + KURE (MSC EUA↔ECSA)
# Regras aplicadas: serviço dominante por TEU (SEASPAN EMPIRE, KURE) e "Múltiplos" para o consórcio do Golfo
GOLFO = "Hapag-Lloyd, MSC, ONE Line, Maersk"
FONTE_T = "Dados técnicos: Flexport Atlas 26/09/2026"
F_GOLFO = ("Serviço Golfo↔ECSA partilhado em 2025 por Hapag-Lloyd (GS1), MSC (Gulf to SAEC String 2), ONE (ANG) "
           "e Maersk (Gulfex/UHA) — rotação publicada (KN, mai/2025) = histórico Atlas; operador que fornece o "
           "navio ao consórcio não identificado → Múltiplos")

# IMO: (nome, TEU, ano, bandeira, LOA, DWT, GT, Beam, reefer_obs, operadores, armador, rota, titular, obs)
NOVOS = {
    9302102: ("RDO ENDEAVOUR", 5642, 2006, "Libéria", 284, 75070, 68570, None, None,
              GOLFO, "Múltiplos", "GS1/UHA/ANG/GULF-SAEC2",
              "Deotrans Schiffahrts (c/o D. Oltmann GmbH & Co KG, Bremen)",
              f"{F_GOLFO} | Atlas Past Port Calls jul-out/2025 no serviço; ANTAQ jan-out com os mesmos pares "
              "(Houston→Santos/Suape, Cartagena↔Santos); saiu para a Ásia após 26/out/2025 (ANTAQ nov-dez = 0) | "
              "Titularidade: MagicPort 26/09/2026 — commercial Oltmann, ISM CPO Containerschiffreederei"),
    9407160: ("SEASPAN EMPIRE", 5041, 2010, "Singapura", 294, 67000, 54940, None, None,
              GOLFO, "Múltiplos", "GS1/UHA/ANG/GULF-SAEC2",
              "Seaspan (Seaspan 1973 Pte Ltd, gestão Seaspan Ship Management, Vancouver)",
              f"{F_GOLFO} | SERVIÇO DOMINANTE POR TEU: Golfo mar-out/2025 = 23.636 TEU (77%); a partir de "
              "6/nov/2025 passou para CON/U4G (Hapag-Lloyd/Maersk) = 7.095 TEU (23%) — atribuídos aqui ao "
              "serviço dominante | Mar-set sem histórico Atlas: serviço inferido pelos pares ANTAQ | "
              "Titularidade: MagicPort 26/09/2026"),
    9085522: ("KURE", 7403, 1996, "Libéria", 318, 96103, 81488, 42.80, 703,
              "MSC", "MSC", "USA-SAEC",
              "Kelsen Shipping Co (c/o Costamare Shipping, Atenas)",
              "Operador: Atlas Past Port Calls jun-ago/2025 = rotação do serviço MSC EUA↔ECSA (mesmo loop do "
              "GREENLAND) | SERVIÇO DOMINANTE POR TEU: abr-ago/2025 = 88% do TEU LC; jan e mar (12%) sem "
              "serviço identificado | Depois de set/2025 fora do Brasil; hoje MSC EMPIRE | Titularidade: MagicPort "
              "26/09/2026 — commercial Costamare, ISM V.Ships Greece | Ficha 21/06/2025: ex-MAERSK KURE, "
              "ex-REGINA MAERSK; TEU nominal 6.418 (Atlas 7.403, mantido Atlas); Beam e reefer (703) da ficha"),
}

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)
assert set(NOVOS) <= set(prio["imo"]), "Há IMOs fora da lista prioritária"

universo = set(tab["Armador_Operador"].dropna()) | {
    o for s in tab["Operadores_Confirmados"].dropna() for o in ops_lista(s)}
em_falta = sorted({o for v in NOVOS.values() for o in ops_lista(v[9])} - universo)
assert not em_falta, f"Rótulos inexistentes na tabela: {em_falta}"
assert "Múltiplos" in set(tab["Armador_Operador"].dropna()), "Rótulo 'Múltiplos' não existe na tabela"

ja = set(NOVOS) & set(tab["IMO"])
if ja == set(NOVOS):
    print(f"Já gravados ({len(tab)} navios) — nada feito.")
elif ja:
    raise AssertionError(f"Estado inesperado: {len(ja)} dos {len(NOVOS)} navios já na tabela")
else:
    print(f"Estado encontrado: tabela sem estes navios ({len(tab)} navios)")
    linhas = []
    for imo, (nome, teu, ano, band, loa, dwt, gt, beam, r_obs, ops, arm, rota, titular, obs) in NOVOS.items():
        lista = ops_lista(ops)
        grupos = {GRUPO_OPS.get(o, o) for o in lista}
        assert arm in lista or arm == "Múltiplos", f"{imo}: Armador_Operador {arm} não está em {lista}"
        assert grupo_titular(titular) not in grupos, f"{imo}: Afretada confirmada com titular dentro dos operadores"

        reefer, metodo = (r_obs, METODO_OBS) if r_obs is not None else reefer_v3(teu)
        outros = [o for o in lista if o != arm]
        linhas.append({
            "IMO": imo, "Nome_Navio": nome, "TEU_Capacidade": teu, "TEU_Reefer_Estimado": reefer,
            "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(teu), "Ano_Construcao": ano,
            "Armador_Operador": arm, "Operadores_Confirmados": ", ".join(lista),
            "Tipo_de_Operacao": "partilhado" if outros else "exclusivo", "Aliança": "Não verificada",
            "Linhas_Partilhadas": ", ".join(outros) if outros else None, "Rota_Servico": rota,
            "Bandeira": band, "LOA": loa, "Beam": beam, "GT": gt, "DWT": dwt,
            "Armador_Titular_Verificado": titular, "Titularidade_Confirmada": True,
            "Status_Titularidade": "Confirmada", "Estrutura_Titularidade": "Afretada",
            "Observacoes_Tecnicas": f"{FONTE_T} | {obs}",
        })
    novos = pd.DataFrame(linhas).reindex(columns=tab.columns)
    nova = (pd.concat([tab, novos.dropna(axis=1, how="all")], ignore_index=True)
              .reindex(columns=tab.columns))
    assert set(tab["IMO"]) <= set(nova["IMO"]) and len(nova) == len(tab) + len(NOVOS)
    falhas = {k: int(m.sum()) for k, m in verificar(nova).items() if m.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)

    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(m.sum()) for k, m in verificar(tab).items() if m.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"

    ganho = prio.loc[prio["imo"].isin(NOVOS), "teu_lc"].sum()
    print(tab.loc[tab["IMO"].isin(NOVOS), ["IMO", "Nome_Navio", "Classe_Navio", "TEU_Reefer_Estimado",
          "Reefer_Metodo", "Armador_Operador", "Tipo_de_Operacao", "Rota_Servico",
          "Estrutura_Titularidade", "Titularidade_Confirmada"]].to_string(index=False))
    print(f"\nGRAVADO — Bloco 13 ({len(NOVOS)} navios) | backup: {bk.name}")
    print(f"  TEU LC destes navios: {ganho:,.0f} (+{ganho / TOTAL_TEU_LC_IMO * 100:.2f} pp)")
    print(f"  {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")
    print("  10 verificações OK no ficheiro gravado")

Já gravados (324 navios) — nada feito.


In [15]:
# Bloco 14 — Correções: LOG-IN RESILIENTE (titularidade) + MSC MICHELA (operador de 2025, regra 0)
UPD = {
    9327669: {"Armador_Titular_Verificado": "MSC (Nendaz Oceanway Ltd c/o Log-In Logística Intermodal, grupo MSC)",
              "Titularidade_Confirmada": True, "Status_Titularidade": "Confirmada",
              "Estrutura_Titularidade": "Afretada"},
    9720512: {"Operadores_Confirmados": "MSC", "Tipo_de_Operacao": "exclusivo",
              "Linhas_Partilhadas": None, "Rota_Servico": "ASIA-ECSA"},
}
OBS = {
    9327669: ("Titularidade corrigida 27/09/2026: MagicPort — registered owner Nendaz Oceanway Ltd c/o Log-In "
              "Logística Intermodal (padrão de SPV MSC), commercial+ISM Log-In → grupo MSC; operadores CMA CGM e "
              "Hapag-Lloyd → Afretada confirmada; pista Döhle (fonte secundária) descartada"),
    9720512: ("Operador de 2025 corrigido 27/09/2026 (regra 0): Atlas Past Port Calls — fora do Brasil jun-nov/2025 "
              "(Índia/Ásia/EUA); no Brasil desde 30/nov/2025 em rotação Ásia-ECSA via Walvis Bay/Khalifa "
              "(semelhante ao Carioca da MSC, nome não confirmado); ANTAQ: abr-mai 13.913 TEU (58%) + dez 10.107 (42%), "
              "pares de carga Ásia/Médio Oriente (Ningbo, Shanghai, Abu Dhabi); Hapag-Lloyd, ZIM e WMSAEC/MSE/SAM são "
              "de 2026 — retirados; abr-mai sem histórico: MSC por ser navio próprio de marca; slots de outras "
              "linhas em 2025 não verificados"),
}

tab = pd.read_csv(PATH_TAB)
assert set(UPD) <= set(tab["IMO"]), "Há IMOs a corrigir que não estão na tabela"
antes = tab.copy()

for imo, campos in UPD.items():
    m = tab["IMO"] == imo
    for col, val in campos.items():
        tab.loc[m, col] = val
    atual = tab.loc[m, "Observacoes_Tecnicas"].iloc[0]
    atual = "" if pd.isna(atual) else str(atual)
    if OBS[imo] not in atual:
        tab.loc[m, "Observacoes_Tecnicas"] = f"{atual} | {OBS[imo]}" if atual else OBS[imo]

# Validação da titularidade nos dois sentidos
def grupos_de(row):
    return {GRUPO_OPS.get(o, o) for o in ops_lista(row["Operadores_Confirmados"])}
r = tab.loc[tab["IMO"] == 9327669].iloc[0]
assert grupo_titular(r["Armador_Titular_Verificado"]) not in grupos_de(r), "RESILIENTE: Afretada com titular dentro dos operadores"
mi = tab.loc[tab["IMO"] == 9720512].iloc[0]
assert grupo_titular(mi["Armador_Titular_Verificado"]) in grupos_de(mi), "MICHELA: Propria com titular fora dos operadores"

if tab.equals(antes):
    print(f"Correções já aplicadas ({len(tab)} navios) — nada feito.")
else:
    falhas = {k: int(v.sum()) for k, v in verificar(tab).items() if v.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"
    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    tab.to_csv(PATH_TAB, index=False)

    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(v.sum()) for k, v in verificar(tab).items() if v.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"
    print(tab.loc[tab["IMO"].isin(UPD), ["IMO", "Nome_Navio", "Armador_Operador", "Operadores_Confirmados",
          "Tipo_de_Operacao", "Rota_Servico", "Estrutura_Titularidade", "Titularidade_Confirmada"]].to_string(index=False))
    print(f"\nGRAVADO — Bloco 14 (2 correções) | backup: {bk.name}")
    print(f"  {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")
    print("  10 verificações OK no ficheiro gravado")

Correções já aplicadas (324 navios) — nada feito.


In [16]:
# Bloco 15 — Fecho das levas: LOG-IN EVOLUTION (nota, opção B) + W KYRENIA (Maersk, extraloader)
FONTE_PRINT = "Econdb"   # trocar se o print do W KYRENIA for de outro site

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)
n0 = len(tab)
mudou = False

# ================= Parte A — LOG-IN EVOLUTION: nota da decisão (operador mantido) =================
IMO_E = 9961960
NOTA_E = ("Decisão de Hugo 27/09/2026 (opção B): mantido Hapag-Lloyd; operador PROVADO só para 12% do TEU LC 2025 "
          "(out+dez, loop NCF/IRL pelo histórico Atlas); jan-fev = 21.140 TEU (88%) em padrão Navegantes↔Rio/Itaguaí "
          "sem histórico — operador não verificado (pista Log-In/MSC); jun-set registado como cabotagem na ANTAQ | "
          "Incluir no teste de sensibilidade; tratamento diferente do ATACAMA (retirado) declarado no README")
m = tab["IMO"] == IMO_E
assert m.sum() == 1, "LOG-IN EVOLUTION não encontrado (ou duplicado) na tabela"
obs = tab.loc[m, "Observacoes_Tecnicas"].iloc[0]
obs = "" if pd.isna(obs) else str(obs)
if NOTA_E in obs:
    print("A) LOG-IN EVOLUTION já tem a nota — nada feito.")
else:
    tab.loc[m, "Observacoes_Tecnicas"] = f"{obs} | {NOTA_E}" if obs else NOTA_E
    mudou = True
    print("A) LOG-IN EVOLUTION: nota acrescentada (operador mantido: Hapag-Lloyd)")

# ================= Parte B — W KYRENIA (IMO 9211494) =================
IMO_K = 9211494
ARM = "Maersk"
TEU, ANO, BANDEIRA = 6802, 2001, "Ilhas Marshall"
LOA, DWT, GT, BEAM = 300, 93552, 80609, 43          # LOA/DWT/GT: MagicPort; TEU e Beam: print
TITULAR = "Maersk (Kauai Shipping Ltd c/o Maersk A/S, Copenhaga)"
OBS_K = (f"Operador: navio próprio Maersk (MagicPort 26/09/2026: registered owner Kauai Shipping Ltd c/o Maersk A/S, "
         f"commercial Maersk A/S, ISM Bernhard Schulte UK) + {FONTE_PRINT} (operador Maersk; serviço atual LAM "
         "Extraloader / Empties Positioning) | ANTAQ 2025: só 2 escalas LC (jul 11.079 TEU, out 12.745 TEU), pares de "
         "carga Santos↔Singapore/Shanghai/Shekou — padrão coerente com extraloader/reposicionamento de vazios | "
         "Serviço de 2025 não verificado por histórico Atlas (sem ficha Atlas) | Dados técnicos: LOA/DWT/GT MagicPort, "
         f"TEU e Beam {FONTE_PRINT} (desvio à regra 'técnicos do Atlas') | Ex-P&O Nedlloyd Shackleton / MAERSK KYRENIA")

assert IMO_K in set(prio["imo"]), "W KYRENIA fora da lista prioritária"
assert ARM in set(tab["Armador_Operador"].dropna()), f"Rótulo '{ARM}' não existe na tabela"

if IMO_K in set(tab["IMO"]):
    print("B) W KYRENIA já na tabela — nada feito.")
else:
    lista = ops_lista(ARM)
    grupos = {GRUPO_OPS.get(o, o) for o in lista}
    assert grupo_titular(TITULAR) in grupos, "Propria confirmada com titular fora dos operadores"
    reefer, metodo = reefer_v3(TEU)
    linha = pd.DataFrame([{
        "IMO": IMO_K, "Nome_Navio": "W KYRENIA", "TEU_Capacidade": TEU,
        "TEU_Reefer_Estimado": reefer, "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(TEU),
        "Ano_Construcao": ANO, "Armador_Operador": ARM, "Operadores_Confirmados": ", ".join(lista),
        "Tipo_de_Operacao": "exclusivo", "Aliança": "Não verificada", "Linhas_Partilhadas": None,
        "Rota_Servico": "EXTRALOADER", "Bandeira": BANDEIRA, "LOA": LOA, "Beam": BEAM, "GT": GT, "DWT": DWT,
        "Armador_Titular_Verificado": TITULAR, "Titularidade_Confirmada": True,
        "Status_Titularidade": "Confirmada", "Estrutura_Titularidade": "Propria",
        "Observacoes_Tecnicas": OBS_K,
    }]).reindex(columns=tab.columns)
    tab = (pd.concat([tab, linha.dropna(axis=1, how="all")], ignore_index=True)
             .reindex(columns=tab.columns))
    mudou = True
    print("B) W KYRENIA inserido (Maersk, Propria confirmada)")

# ================= Gravação única =================
if not mudou:
    print(f"\nNada a gravar ({len(tab)} navios).")
else:
    falhas = {k: int(v.sum()) for k, v in verificar(tab).items() if v.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"
    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    tab.to_csv(PATH_TAB, index=False)
    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(v.sum()) for k, v in verificar(tab).items() if v.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"
    print(f"\nGRAVADO — Bloco 15 | backup: {bk.name} | {len(tab)} navios")

A) LOG-IN EVOLUTION já tem a nota — nada feito.
B) W KYRENIA já na tabela — nada feito.

Nada a gravar (324 navios).


In [17]:
# Bloco 16 — Titularidade MSC confirmada por decisão do autor: MSC SILVIA + MSC ELLEN X
UPD = {
    9720457: ("SILVIA", "MSC (registered owner Xinlong International Ship, Hong Kong — titularidade MSC por decisão do autor)",
              "Titularidade confirmada por decisão do autor 27/09/2026: registered owner Xinlong International Ship com "
              "morada só em Hong Kong (sem c/o MSC); outras bases indicam owner MSC; opera exclusivamente para a MSC "
              "(MALABAR) | Prova mais fraca que as SPVs c/o MSC — declarado no README"),
    9166780: ("ELLEN", "MSC (Soroe Shipping, SPV de 1 navio — titularidade MSC por decisão do autor)",
              "Titularidade confirmada por decisão do autor 27/09/2026: SPV Soroe Shipping de 1 navio com morada c/o "
              "Maersk (provavelmente desatualizada: ex-SOROE MAERSK até abr/2021); ISM e commercial MSC; fontes "
              "secundárias indicam owner MSC | Declarado no README"),
}

tab = pd.read_csv(PATH_TAB)
antes = tab.copy()

for imo, (chave_nome, titular, nota) in UPD.items():
    m = tab["IMO"] == imo
    assert m.sum() == 1, f"{imo} não encontrado (ou duplicado) na tabela"
    nome = str(tab.loc[m, "Nome_Navio"].iloc[0])
    assert chave_nome in nome.upper(), f"{imo} é '{nome}', não o navio esperado — nada foi alterado"
    assert tab.loc[m, "Estrutura_Titularidade"].iloc[0] == "Propria", f"{nome}: estrutura não é Propria"
    ops = tab.loc[m, "Operadores_Confirmados"].iloc[0]
    grupos = {GRUPO_OPS.get(o, o) for o in ops_lista(ops)}
    g_tit = grupo_titular(titular)
    assert g_tit in grupos, (f"{nome}: titular fora do grupo dos operadores — "
                             f"grupo_titular='{g_tit}', operadores='{ops}', grupos={grupos}")

    tab.loc[m, "Armador_Titular_Verificado"] = titular
    tab.loc[m, "Titularidade_Confirmada"] = True
    tab.loc[m, "Status_Titularidade"] = "Confirmada"
    obs = tab.loc[m, "Observacoes_Tecnicas"].iloc[0]
    obs = "" if pd.isna(obs) else str(obs)
    if nota not in obs:
        tab.loc[m, "Observacoes_Tecnicas"] = f"{obs} | {nota}" if obs else nota

if tab.equals(antes):
    print(f"Titularidades já aplicadas ({len(tab)} navios) — nada feito.")
else:
    falhas = {k: int(v.sum()) for k, v in verificar(tab).items() if v.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"
    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    tab.to_csv(PATH_TAB, index=False)

    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(v.sum()) for k, v in verificar(tab).items() if v.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"
    print(tab.loc[tab["IMO"].isin(UPD), ["IMO", "Nome_Navio", "Armador_Operador", "Estrutura_Titularidade",
          "Titularidade_Confirmada", "Armador_Titular_Verificado"]].to_string(index=False))
    print(f"\nGRAVADO — Bloco 16 (2 titularidades) | backup: {bk.name}")
    print(f"  {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")
    print("  10 verificações OK no ficheiro gravado")

Titularidades já aplicadas (324 navios) — nada feito.


In [18]:
# Bloco 17 — Para os 90%: MSC MAGNUM VII + MSC NERISSA V (serviço MSC EUA↔ECSA, regra 0)
FONTE_T = "Dados técnicos: Flexport Atlas 27/09/2026"
F_MSC = ("Operador: Atlas Past Port Calls 2025 — escalas no Brasil só no serviço MSC EUA↔ECSA (Santos, Buenos Aires, "
         "Montevideo, Paranaguá, Itajaí, Rio Grande, Rio, Salvador, Colón, Freeport, costa leste EUA), o mesmo loop "
         "do KURE e do GREENLAND | Serviços de 2026 NÃO usados")

# IMO: (nome, TEU, ano, bandeira, LOA, DWT, GT, Beam, reefer_obs, titular, confirmada, obs)
NOVOS = {
    9252541: ("MSC MAGNUM VII", 6732, 2003, "Libéria", 299, 97557, 75590, None, None,
              "MSC (Aquata Navigation Ltd c/o MSC Shipmanagement Limassol)", True,
              f"{F_MSC} | Jun-ago/2025 no Mediterrâneo↔costa leste EUA (fora do Brasil); Brasil set-dez/2025; "
              "jan-mai/2025 sem histórico — MSC por ser navio próprio da marca | Atlas 2026: Maersk UHD + MSC "
              "GULF-SAEC1 (não usado) | Titularidade: MagicPort 27/09/2026 — commercial e ISM MSC Shipmanagement"),
    9278155: ("MSC NERISSA V", 5059, 2004, "Panamá", 294, 72902, 54881, 32.25, 400,
              None, False,
              f"{F_MSC} | Jul/2025 Médio Oriente/Índia; Brasil set-out e nov-dez/2025 | Titularidade NÃO confirmada: "
              "só ficha técnica 20/07/2020 (ainda como MSC NERISSA) — registered owner Altair Carriers Company SA "
              "(Panamá, sem morada c/o MSC), manager MSC Shipmanagement, operador MSC; renomeado NERISSA V (sufixo "
              "pode indicar mudança de dono); falta MagicPort | Beam e reefer (400) da ficha 2020; TEU ficha 5.048 vs "
              "Atlas 5.059 (mantido Atlas) | Fisicamente Panamax (Beam 32,25 m), classe Neo-Panamax pela regra de TEU"),
}

tab = pd.read_csv(PATH_TAB)
prio = pd.read_csv(PATH_PRIO)
assert "MSC" in set(tab["Armador_Operador"].dropna()), "Rótulo 'MSC' não existe na tabela"

ja = set(NOVOS) & set(tab["IMO"])
if ja == set(NOVOS):
    print(f"Bloco 17 já aplicado ({len(tab)} navios) — nada feito.")
elif ja:
    raise AssertionError(f"Estado inesperado: {len(ja)} dos {len(NOVOS)} navios já na tabela")
else:
    print(f"Estado encontrado: tabela sem estes navios ({len(tab)} navios)")
    lista = ops_lista("MSC")
    grupos = {GRUPO_OPS.get(o, o) for o in lista}
    linhas = []
    for imo, (nome, teu, ano, band, loa, dwt, gt, beam, r_obs, titular, conf, obs) in NOVOS.items():
        if conf:
            assert grupo_titular(titular) in grupos, f"{imo}: Propria confirmada com titular fora do grupo MSC"
        reefer, metodo = (r_obs, METODO_OBS) if r_obs is not None else reefer_v3(teu)
        linhas.append({
            "IMO": imo, "Nome_Navio": nome, "TEU_Capacidade": teu, "TEU_Reefer_Estimado": reefer,
            "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(teu), "Ano_Construcao": ano,
            "Armador_Operador": "MSC", "Operadores_Confirmados": "MSC",
            "Tipo_de_Operacao": "exclusivo", "Aliança": "Não verificada", "Linhas_Partilhadas": None,
            "Rota_Servico": "USA-SAEC", "Bandeira": band, "LOA": loa, "Beam": beam, "GT": gt, "DWT": dwt,
            "Armador_Titular_Verificado": titular, "Titularidade_Confirmada": conf,
            "Status_Titularidade": "Confirmada" if conf else "Não confirmada",
            "Estrutura_Titularidade": "Propria",
            "Observacoes_Tecnicas": f"{FONTE_T} | {obs}",
        })
    novos = pd.DataFrame(linhas).reindex(columns=tab.columns)
    nova = (pd.concat([tab, novos.dropna(axis=1, how="all")], ignore_index=True)
              .reindex(columns=tab.columns))
    assert set(tab["IMO"]) <= set(nova["IMO"]) and len(nova) == len(tab) + len(NOVOS)
    falhas = {k: int(m.sum()) for k, m in verificar(nova).items() if m.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)
    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(m.sum()) for k, m in verificar(tab).items() if m.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"

    ganho = prio.loc[prio["imo"].isin(NOVOS), "teu_lc"].sum()
    print(tab.loc[tab["IMO"].isin(NOVOS), ["IMO", "Nome_Navio", "Classe_Navio", "TEU_Reefer_Estimado",
          "Reefer_Metodo", "Armador_Operador", "Rota_Servico", "Estrutura_Titularidade",
          "Titularidade_Confirmada"]].to_string(index=False))
    print(f"\nGRAVADO — Bloco 17 ({len(NOVOS)} navios) | backup: {bk.name}")
    print(f"  TEU LC (prio): {ganho:,.0f} — se aparecer 0, estes IMOs não estão no imos_prioritarios.csv; "
          "o valor certo vem da célula de cobertura")
    print(f"  {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")
    print("  10 verificações OK no ficheiro gravado")

Bloco 17 já aplicado (324 navios) — nada feito.


In [19]:
# Bloco 18 — Titularidade confirmada: MSC NERISSA V (MagicPort 28/09/2026)
IMO_N = 9278155
TITULAR = "MSC (Compania Naviera Lasma SA c/o MSC Mediterranean Shipping Co, Genebra)"
NOTA = ("Titularidade confirmada 28/09/2026: MagicPort — registered owner Compania Naviera Lasma SA c/o MSC Genebra, "
        "commercial MSC Mediterranean Shipping Co, ISM MSC Shipmanagement | Em 2020 o owner era Altair Carriers "
        "Company SA (ficha 20/07/2020): mudança de dono coerente com o renome NERISSA → NERISSA V")

tab = pd.read_csv(PATH_TAB)
m = tab["IMO"] == IMO_N
assert m.sum() == 1, "MSC NERISSA V não encontrado (ou duplicado) na tabela"
assert "NERISSA" in str(tab.loc[m, "Nome_Navio"].iloc[0]).upper(), "O IMO não corresponde ao MSC NERISSA V"
assert tab.loc[m, "Estrutura_Titularidade"].iloc[0] == "Propria", "Estrutura não é Propria"
grupos = {GRUPO_OPS.get(o, o) for o in ops_lista(tab.loc[m, "Operadores_Confirmados"].iloc[0])}
assert grupo_titular(TITULAR) in grupos, f"Titular fora do grupo dos operadores: {grupo_titular(TITULAR)} vs {grupos}"

antes = tab.copy()
tab.loc[m, "Armador_Titular_Verificado"] = TITULAR
tab.loc[m, "Titularidade_Confirmada"] = True
tab.loc[m, "Status_Titularidade"] = "Confirmada"
obs = tab.loc[m, "Observacoes_Tecnicas"].iloc[0]
obs = "" if pd.isna(obs) else str(obs)
if NOTA not in obs:
    tab.loc[m, "Observacoes_Tecnicas"] = f"{obs} | {NOTA}" if obs else NOTA

if tab.equals(antes):
    print(f"Bloco 18 já aplicado ({len(tab)} navios) — nada feito.")
else:
    falhas = {k: int(v.sum()) for k, v in verificar(tab).items() if v.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"
    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    tab.to_csv(PATH_TAB, index=False)
    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(v.sum()) for k, v in verificar(tab).items() if v.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"
    print(f"GRAVADO — Bloco 18 | backup: {bk.name} | {len(tab)} navios | "
          f"confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")

Bloco 18 já aplicado (324 navios) — nada feito.


In [20]:
# Bloco 19 — Cauda: os 4 maiores navios não mapeados (verificação da liderança Maersk vs MSC)
FONTE_T = "Dados técnicos: Flexport Atlas 28/09/2026"
MP = "Titularidade: MagicPort 28/09/2026"

# IMO: (nome, TEU, ano, bandeira, LOA, DWT, GT, Beam, reefer_obs, operadores, armador, rota, titular, estrutura, obs)
NOVOS = {
    9525376: ("MAERSK CHAMBAL", 4496, 2012, "Hong Kong", 249, 61614, 50714, None, None,
              "Maersk, MSC", "Maersk", "UHD/GULF-SAEC1",
              "Maersk (Maersk Shipping Hong Kong Ltd c/o Maersk A/S)", "Propria",
              f"Operador (regra 0): ANTAQ só out e dez/2025 (8.374 + 7.157 TEU), 100% cobertos pelo histórico Atlas no "
              "loop Golfo↔ECSA (Houston, Veracruz, Cartagena, Santos, Itapoá, Paranaguá, Rio) = Maersk UHD + MSC "
              "GULF-SAEC1, o mesmo em 2026 | Jun-ago/2025 no Norte da Europa/Báltico | {MP} — commercial Moller "
              "Singapore AP, ISM Maersk A/S"),
    9224300: ("TINCAN EXPRESS", 4253, 2001, "Singapura", 259, 50790, 39892, 32.35, 400,
              "Hapag-Lloyd, Maersk", "Hapag-Lloyd", "CON/U4G",
              "Seaspan (Seaspan 1342 Pte Ltd, gestão Seaspan Ship Management, Vancouver)", "Afretada",
              "Operador INFERIDO (trade + marca, sem histórico Atlas): ANTAQ só jan-jun/2025, pares Santos→Callao 16%, "
              "San Antonio, Guayaquil = mesmo loop ECSA↔WCSA do DUSSELDORF EXPRESS (Hapag CON + Maersk U4G, confirmado "
              "por histórico); nome 'Express' = padrão Hapag para fretamentos longos | Parceiro em jan/2025 (antes da "
              "Gemini) não verificado | Histórico Atlas só desde set/2025, sem Brasil; hoje Hapag WC1 | "
              f"{MP} | Beam e reefer (400): ficha 06/10/2015 (ex-SEASPAN HAMBURG, CSAV LICANTEN, CSCL HAMBURG)"),
    9709207: ("CMA CGM MARSEILLE", 2140, 2015, "Malta", 190, 27322, 26645, None, None,
              "CMA CGM, Marfret", "CMA CGM", "NEFGUI/SAS",
              "CMA CGM (Guyane 4093 c/o CMA CGM SA, Marselha)", "Propria",
              "Operadores: Atlas 2026 (CMA CGM NEFGUI + Marfret SAS) validados pelo histórico — o mesmo loop "
              "Norte da Europa↔Caraíbas/Guiana↔Vila do Conde/Fortaleza de jun/2025 a set/2026; ANTAQ fev-set/2025 "
              f"(pares Fortaleza↔Rotterdam/London/Vigo/Port of Spain) | {MP} — commercial CMA CGM, ISM CMA Ships"),
    9709219: ("CMA CGM SAINT LAURENT", 2140, 2015, "França", 190, 27322, 26645, None, None,
              "CMA CGM, Marfret", "CMA CGM", "NEFGUI/SAS",
              "CMA CGM (Guyane 4094 c/o CMA CGM SA, Marselha)", "Propria",
              "Operadores: Atlas 2026 (CMA CGM NEFGUI + Marfret SAS) validados pelo histórico — o mesmo loop "
              "de jun/2025 a set/2026; ANTAQ jan-dez/2025 com o mesmo padrão (Fortaleza↔Rotterdam/Vigo/London) | "
              f"{MP} — commercial CMA CGM, ISM CMA Ships | MagicPort lista como CMA CGM ST. LAURENT"),
}

tab = pd.read_csv(PATH_TAB)
universo = set(tab["Armador_Operador"].dropna()) | {
    o for s in tab["Operadores_Confirmados"].dropna() for o in ops_lista(s)}
em_falta = sorted({o for v in NOVOS.values() for o in ops_lista(v[9])} - universo)
assert not em_falta, f"Rótulos inexistentes na tabela: {em_falta}"

ja = set(NOVOS) & set(tab["IMO"])
if ja == set(NOVOS):
    print(f"Bloco 19 já aplicado ({len(tab)} navios) — nada feito.")
elif ja:
    raise AssertionError(f"Estado inesperado: {len(ja)} dos {len(NOVOS)} navios já na tabela")
else:
    print(f"Estado encontrado: tabela sem estes navios ({len(tab)} navios)")
    linhas = []
    for imo, (nome, teu, ano, band, loa, dwt, gt, beam, r_obs, ops, arm, rota, titular, estr, obs) in NOVOS.items():
        lista = ops_lista(ops)
        grupos = {GRUPO_OPS.get(o, o) for o in lista}
        assert arm in lista, f"{imo}: Armador_Operador {arm} não está em {lista}"
        if estr == "Propria":
            assert grupo_titular(titular) in grupos, f"{imo}: Propria com titular fora dos operadores ({grupo_titular(titular)})"
        else:
            assert grupo_titular(titular) not in grupos, f"{imo}: Afretada com titular dentro dos operadores"
        reefer, metodo = (r_obs, METODO_OBS) if r_obs is not None else reefer_v3(teu)
        outros = [o for o in lista if o != arm]
        linhas.append({
            "IMO": imo, "Nome_Navio": nome, "TEU_Capacidade": teu, "TEU_Reefer_Estimado": reefer,
            "Reefer_Metodo": metodo, "Classe_Navio": classe_navio(teu), "Ano_Construcao": ano,
            "Armador_Operador": arm, "Operadores_Confirmados": ", ".join(lista),
            "Tipo_de_Operacao": "partilhado" if outros else "exclusivo", "Aliança": "Não verificada",
            "Linhas_Partilhadas": ", ".join(outros) if outros else None, "Rota_Servico": rota,
            "Bandeira": band, "LOA": loa, "Beam": beam, "GT": gt, "DWT": dwt,
            "Armador_Titular_Verificado": titular, "Titularidade_Confirmada": True,
            "Status_Titularidade": "Confirmada", "Estrutura_Titularidade": estr,
            "Observacoes_Tecnicas": f"{FONTE_T} | {obs}",
        })
    novos = pd.DataFrame(linhas).reindex(columns=tab.columns)
    nova = (pd.concat([tab, novos.dropna(axis=1, how="all")], ignore_index=True)
              .reindex(columns=tab.columns))
    assert set(tab["IMO"]) <= set(nova["IMO"]) and len(nova) == len(tab) + len(NOVOS)
    falhas = {k: int(v.sum()) for k, v in verificar(nova).items() if v.sum()}
    assert not falhas, f"Verificações falharam — nada foi gravado: {falhas}"

    bk = BACKUPS / f"tabela_completa_backup_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(PATH_TAB, bk)
    nova.to_csv(PATH_TAB, index=False)
    tab = pd.read_csv(PATH_TAB)
    falhas = {k: int(v.sum()) for k, v in verificar(tab).items() if v.sum()}
    assert not falhas, f"Falhas no ficheiro gravado (repor {bk.name}): {falhas}"
    print(tab.loc[tab["IMO"].isin(NOVOS), ["IMO", "Nome_Navio", "Classe_Navio", "TEU_Reefer_Estimado",
          "Armador_Operador", "Operadores_Confirmados", "Estrutura_Titularidade"]].to_string(index=False))
    print(f"\nGRAVADO — Bloco 19 ({len(NOVOS)} navios) | backup: {bk.name}")
    print(f"  {len(tab)} navios | confirmados: {int(tab['Titularidade_Confirmada'].sum())} | "
          f"{tab['Estrutura_Titularidade'].value_counts().to_dict()}")
    print("  10 verificações OK no ficheiro gravado")

Bloco 19 já aplicado (324 navios) — nada feito.


In [21]:
# Célula de cobertura (SÓ LEITURA) — substitui a antiga; tudo derivado dos ficheiros, sem listas de navios à mão
import unicodedata, re
from pathlib import Path

RAW = BASE / "data" / "raw"
PATH_DESC = Path(PATH_TAB).parent / "navios_descartados.csv"
DECLARADOS = {9718947: "ATACAMA — sem prova de operador em 2025 (escalas BR só jan-abr, antes do histórico Atlas)"}
TAURUS = 9247766

def norm(c):
    c = unicodedata.normalize("NFKD", str(c)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", c).strip("_")

if "cargo" not in globals():   # evita reler o .xlsx se já está em memória
    cargo = pd.read_excel(RAW / "antaq_2025_cargo.xlsx"); cargo.columns = [norm(c) for c in cargo.columns]
TEU_C = "total_de_movimentacao_conteineres_bruto_teu_twenty_feet_equivalent_unit"

# ---------- 1) TEU LC por IMO, recalculado da ANTAQ ----------
lc = cargo[cargo["navegacao_da_atracacao"].astype(str).str.lower().str.contains("longo")].copy()
lc["imo"] = pd.to_numeric(lc["imo_da_embarcacao"], errors="coerce")
teu_imo = lc.dropna(subset=["imo"]).groupby("imo")[TEU_C].sum()
teu_imo.index = teu_imo.index.astype(int)
total = teu_imo.sum()
print(f"TEU LC com IMO recalculado: {total:,.1f} ({len(teu_imo)} IMOs) | referência do 02: {TOTAL_TEU_LC_IMO:,.1f} "
      f"| diferença: {total - TOTAL_TEU_LC_IMO:,.1f}")

# ---------- 2) Categorias ----------
tab = pd.read_csv(PATH_TAB)
desc = pd.read_csv(PATH_DESC)
em_tab, em_desc, em_decl = set(tab["IMO"]), set(desc["IMO"]), set(DECLARADOS)

assert not (em_tab & em_desc), f"IMOs na tabela E nos descartados: {em_tab & em_desc}"
assert not (em_tab & em_decl), f"IMOs na tabela E nos pendentes declarados: {em_tab & em_decl}"

cat = pd.Series("cauda", index=teu_imo.index)
cat[cat.index.isin(em_tab)] = "tabela"
cat[cat.index.isin(em_desc)] = "descartado"
cat[cat.index.isin(em_decl)] = "pendente declarado"

res = (pd.DataFrame({"categoria": cat, "teu": teu_imo}).groupby("categoria")
         .agg(IMOs=("teu", "size"), TEU_LC=("teu", "sum")))
res["% do total"] = (res["TEU_LC"] / total * 100).round(2)
res = res.reindex(["tabela", "pendente declarado", "descartado", "cauda"]).fillna(0)
print("\n" + res.to_string(float_format=lambda x: f"{x:,.1f}"))
print(f"Soma das categorias: {res['TEU_LC'].sum():,.1f} (tem de ser igual ao total: {total:,.1f})")

cob = res.loc["tabela", "TEU_LC"]
print(f"\nCOBERTURA: {cob / total * 100:.2f}% do TEU LC recalculado | {cob / TOTAL_TEU_LC_IMO * 100:.2f}% da referência do 02")

# ---------- 3) Verificações de coerência ----------
sem_teu = sorted(em_tab - set(teu_imo.index))
print(f"\nNavios da tabela sem TEU LC na ANTAQ: {len(sem_teu)}" + (f" → {sem_teu[:15]}" if sem_teu else ""))
print(f"MSC TAURUS VII ({TAURUS}) na tabela: {'SIM' if TAURUS in em_tab else 'NÃO — verificar'}")
print(f"Tabela: {len(tab)} navios | descartados: {len(desc)} | pendentes declarados: {len(DECLARADOS)}")

# ---------- 4) Os 10 maiores da cauda ----------
cauda = teu_imo[cat == "cauda"].sort_values(ascending=False)
print(f"\nCauda: {len(cauda)} IMOs | os 10 maiores (TEU LC e pp):")
print((pd.DataFrame({"TEU_LC": cauda.head(10), "pp": (cauda.head(10) / total * 100).round(3)})
         .to_string(float_format=lambda x: f"{x:,.3f}")))

TEU LC com IMO recalculado: 11,085,401.9 (675 IMOs) | referência do 02: 11,085,401.9 | diferença: 0.0

                    IMOs       TEU_LC  % do total
categoria                                        
tabela               324 10,037,246.2        90.5
pendente declarado     1     24,079.8         0.2
descartado             1     20,374.2         0.2
cauda                349  1,003,701.7         9.1
Soma das categorias: 11,085,401.9 (tem de ser igual ao total: 11,085,401.9)

COBERTURA: 90.54% do TEU LC recalculado | 90.54% da referência do 02

Navios da tabela sem TEU LC na ANTAQ: 0
MSC TAURUS VII (9247766) na tabela: SIM
Tabela: 324 navios | descartados: 1 | pendentes declarados: 1

Cauda: 349 IMOs | os 10 maiores (TEU LC e pp):
            TEU_LC    pp
imo                     
9450612 14,723.250 0.133
9603233 14,626.000 0.132
9464699 14,603.750 0.132
9344643 14,524.000 0.131
9997919 14,426.000 0.130
9996018 13,944.000 0.126
9745653 13,835.000 0.125
9976434 13,754.000 0.124
9997921 13

In [22]:
# Ferramenta (SÓ LEITURA) — padrão de escalas ANTAQ 2025 por IMO: meses, TEU LC e pares origem→destino da CARGA
# Uso: trocar os IMOs em IMOS e correr. Não grava nada. Nota: os pares são origem/destino da carga,
# não a próxima escala do navio.
import unicodedata, re
from pathlib import Path

IMOS = {9525376: "MAERSK CHAMBAL", 9224300: "TINCAN EXPRESS", 9709207: "CMA CGM MARSEILLE", 9709219: "CMA CGM SAINT LAURENT"}

RAW = BASE / "data" / "raw"
F_CARGO, F_PORTS = RAW / "antaq_2025_cargo.xlsx", RAW / "antaq_2025_cargo_ports.xlsx"
assert F_CARGO.exists() and F_PORTS.exists(), f"Ficheiros não encontrados em {RAW}"

def norm(c):
    c = unicodedata.normalize("NFKD", str(c)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9]+", "_", c).strip("_")

if "cargo" not in globals():
    cargo = pd.read_excel(F_CARGO); cargo.columns = [norm(c) for c in cargo.columns]
if "ports" not in globals():
    ports = pd.read_excel(F_PORTS); ports.columns = [norm(c) for c in ports.columns]

TEU_C = "total_de_movimentacao_conteineres_bruto_teu_twenty_feet_equivalent_unit"
TEU_P = TEU_C if TEU_C in ports.columns else next(c for c in ports.columns if "teu" in c)
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]

lc = cargo[cargo["navegacao_da_atracacao"].astype(str).str.lower().str.contains("longo")].copy()
lc["imo"] = pd.to_numeric(lc["imo_da_embarcacao"], errors="coerce")
lc["m"] = lc["mes"].astype(str).str.lower().str[:3]

for imo, nome in IMOS.items():
    s = lc[lc["imo"] == imo]
    print(f"\n=== {imo} {nome} — TEU LC {s[TEU_C].sum():,.0f} "
          f"({s[TEU_C].sum() / TOTAL_TEU_LC_IMO * 100:.2f} pp) | atracações {s['id_atracacao'].nunique()} ===")
    if s.empty:
        print("  sem escalas de Longo Curso no ficheiro")
        continue
    mes = (s.groupby("m").agg(atrac=("id_atracacao", "nunique"), teu=(TEU_C, "sum"))
             .reindex(MESES).fillna(0).astype(int))
    print(mes.T.to_string())
    p = ports[ports["id_atracacao"].isin(s["id_atracacao"].unique())]
    par = (p.groupby(["instalacao_de_origem", "instalacao_de_destino"])[TEU_P].sum()
             .sort_values(ascending=False))
    tot = par.sum()
    print("  Top pares origem → destino da CARGA (% do TEU):")
    print((par.head(6) / tot * 100).round(1).to_string() if tot else "  sem TEU no cargo_ports")


=== 9525376 MAERSK CHAMBAL — TEU LC 15,531 (0.14 pp) | atracações 10 ===
m      jan  fev  mar  abr  mai  jun  jul  ago  set   out  nov   dez
atrac    0    0    0    0    0    0    0    0    0     5    0     5
teu      0    0    0    0    0    0    0    0    0  8374    0  7157
  Top pares origem → destino da CARGA (% do TEU):
instalacao_de_origem               instalacao_de_destino            
Santos                             Porto Itapoá Terminais Portuários    11.1
Porto Itapoá Terminais Portuários  Cartagena                             7.8
Santos                             Cartagena                             6.4
Cartagena                          Porto Itapoá Terminais Portuários     6.2
                                   Santos                                4.3
Paranaguá                          Veracruz                              4.0

=== 9224300 TINCAN EXPRESS — TEU LC 15,170 (0.14 pp) | atracações 12 ===
m       jan   fev   mar  abr   mai   jun  jul  ago  set  out  nov  